In [ ]:
# Imports ────────────────────────────────────────────────────────────────────
from __future__ import annotations
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
import spyndex
import ruptures as rpt
from scipy.signal import savgol_filter

# Config ────────────────────────────────────────────────────────────────────
TARGET_YEAR       = 2025
TOWER_X, TOWER_Y  = 395509.71, 5309956.22          # EPSG:32632 (UTM 32N)
ROI_RADIUS_M      = 100.0                           # satellite ROI

# Satellite processing
S1_ORBIT          = "ascending"                     # "ascending" | "descending"
RH_THRESH_PCT     = 95.0                            # drop S1 scenes with RH > thresh at acquisition
S1_RAIN_THRESH_MM = 1.0                             # drop S1 scenes with >1 mm in 12 h prior
S1_THETA_REF      = 40.0                            # cosine-θ normalisation reference
S1_N              = 2                               # cos-θ exponent (vegetation: 2)
S1_SMOOTH_WINDOW  = 2                               # rolling median over ~3 revisits

# Changepoint / stress
SWP_STRESS_MPA     = -1.5                           # physiological stress threshold
ZOOM_BUFFER_DAYS   = 21                             # default zoom buffer
BUFFER_SWEEP_DAYS  = (14, 21, 30)                   # sensitivity sweep for dense streams
BASELINE_DAYS_PRE  = (30, 7)                        # [stress_start-30d, stress_start-7d] for dense
BASELINE_DAYS_PRE_SPARSE = (90, 7)                  # wider window for sparse streams (S1/S2/TLS)
DROP_BASELINE_MIN_SPARSE = 3                        # sparse streams need fewer baseline obs
SIGMA_THRESH       = 2.0                            # sparse-stream departure threshold
SIGMA_RECOVER      = 1.0                            # sparse-stream recovery threshold
DROP_NOBS_MIN      = 4                              # drop streams with fewer obs in zoom
DROP_BASELINE_MIN  = 5                              # drop streams with fewer obs in baseline
DROP_GAPFRAC_MAX   = 0.10                           # drop dense streams with > 10% gaps

# Leaf angle / PAI
LEAF_ANGLE_WINDOW_UTC = (11, 15)                    # daytime window (UTC hours)
LEAF_PRECIP_THR_MM    = 0.0001                      # precip trigger for leaf quality mask
LEAF_PRECIP_LOCKOUT_H = 6.0                        # lockout hours after wet step
UNDERSTORY_MAX_M      = 10.0                        # PAI split height

# Tree / camera assignments (h10546 is the SWP + cam65/66/67 tree)
SWP_TREE           = "h10546"
LEAF_CAMS          = ["cam65", "cam66", "cam67"]    # for h10546

# Meteorological stress thresholds
DRY_RUN_MIN_DAYS   = 10                             # consecutive dry days
DRY_PRECIP_MAX_MM  = 0.5                            # a day is "dry" below this
VPD_HIGH_HPA       = 15.0                           # daily-max VPD threshold for stress
SM_LOW_PCT         = 20.0                           # root-zone SM threshold for stress

# Sentinel-2: manually-curated clear-sky dates for 2025
S2_MANUAL_DATES_2025 = [
    "2025-01-13","2025-02-27","2025-03-04","2025-03-09","2025-03-19",
    "2025-04-03","2025-04-08","2025-04-10","2025-04-28","2025-04-30",
    "2025-05-10","2025-05-20","2025-05-30","2025-06-09","2025-06-12",
    "2025-06-17","2025-06-19","2025-06-22","2025-06-29","2025-07-02",
    "2025-08-08","2025-08-11","2025-08-18","2025-08-26","2025-08-31",
    "2025-09-20","2025-10-07","2025-10-15","2025-10-30","2025-11-04",
    "2025-11-06","2025-11-19","2025-12-09",
]
S2_INDICES = ["NDVI", "kNDVI", "EVI", "NIRv", "SAVI", "CIRE", "MTCI", "NDII"]
S1_LINEAR_VARS = ["VV_lin", "VH_lin", "SPAN_lin", "CR_lin", "RVI"]

# Paths ─────────────────────────────────────────────────────────────────────
ROOT_DIR = Path.cwd().parents[2]                    # notebooks/02_analysis/poster/ → project root
DATA_DIR = ROOT_DIR / "data"
FIG_DIR  = ROOT_DIR / "figures" / "poster"
FIG_DIR.mkdir(parents=True, exist_ok=True)

P = {
    "meteo":   DATA_DIR / "processed/atmosphere_soil/meteo_dehar_30min.csv",
    "fluxes":  DATA_DIR / "processed/atmosphere_soil/fluxes_dehar_30min.csv",
    "sm":      DATA_DIR / "processed/atmosphere_soil/soil_moisture_dehar_30min.csv",
    "precip":  DATA_DIR / "processed/atmosphere_soil/HARTHM_2025_Precipitation_30min_UTC.csv",
    "sapflow": DATA_DIR / "processed/physiology/sap_flux_density/sapflow_dehar_30min.csv",
    "swp":     DATA_DIR / "processed/physiology/stemwater_potential/swp_dehar_15min.csv",
    "twd":     DATA_DIR / "processed/physiology/twd/twd_dehar_30min.csv",
    "vod":     DATA_DIR / "processed/proximal_rs/gnss_vod/gnss_vod_dehar_30min.csv",
    "leaf_angle": DATA_DIR / "processed/proximal_rs/anglecam/leaf_angle_cam65_66_67_native.csv",
    "pai":     DATA_DIR / "processed/proximal_rs/leaf/leaf_hemi_hi_2025_scan05utc.csv",
    "s1_dir":  DATA_DIR / "processed/satellite/sentinel1",
    "s2_dir":  DATA_DIR / "processed/satellite/sentinel2",
}

# Paul Tol palettes (colour-blind-safe) ─────────────────────────────────────
TOL_BRIGHT = {
    "blue":   "#4477AA", "red":    "#EE6677", "green":  "#228833",
    "yellow": "#CCBB44", "cyan":   "#66CCEE", "purple": "#AA3377",
    "grey":   "#BBBBBB", "black":  "#000000",
}
TOL_LIGHT = {
    "lblue":  "#77AADD", "lcyan":  "#99DDFF", "lmint":  "#44BB99",
    "lpear":  "#BBCC33", "lolive": "#AAAA00", "lyellow":"#EEDD88",
    "lorange":"#EE8866", "lpink":  "#FFAABB", "lgrey":  "#DDDDDD",
}
# Domain colour mapping for Figure 2 Gantt chart
DOMAIN_COLOR = {
    "atmosphere":  TOL_BRIGHT["red"],
    "soil":        TOL_BRIGHT["yellow"],
    "physiology":  TOL_BRIGHT["green"],
    "proximal":    TOL_BRIGHT["cyan"],
    "satellite":   TOL_BRIGHT["purple"],
    "flux":        TOL_BRIGHT["blue"],
}

# Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="ticks", font_scale=1.0)
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "axes.grid":  True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.5,
    "grid.linestyle": "--",
    "figure.dpi": 110,
})

P = {
    "meteo":      DATA_DIR / "processed/atmosphere_soil/meteo_dehar_30min.csv",
    "fluxes":     DATA_DIR / "processed/atmosphere_soil/fluxes_dehar_30min.csv",
    "sm":         DATA_DIR / "processed/atmosphere_soil/soil_moisture_dehar_30min.csv",
    "precip":     DATA_DIR / "processed/atmosphere_soil/HARTHM_2025_Precipitation_30min_UTC.csv",
    "sapflow":    DATA_DIR / "processed/physiology/sap_flux_density/sapflow_dehar_30min.csv",
    "swp":        DATA_DIR / "processed/physiology/stemwater_potential/swp_dehar_15min.csv",
    "twd":        DATA_DIR / "processed/physiology/twd/twd_dehar_30min.csv",
    "vod":        DATA_DIR / "processed/proximal_rs/gnss_vod/gnss_vod_dehar_30min.csv",
    "leaf_angle": DATA_DIR / "processed/proximal_rs/anglecam/leaf_angle_cam60_70_native.csv",
    "pai":        DATA_DIR / "processed/proximal_rs/leaf/leaf_hemi_hi_2025_scan05utc.csv",
    "s1_dir":     DATA_DIR / "processed/satellite/sentinel1",
    "s2_dir":     DATA_DIR / "processed/satellite/sentinel2",
}

STRESS_SHADE_COLOR = "#E76F51"
print(f"Target year: {TARGET_YEAR}  |  Root: {ROOT_DIR}")

In [ ]:
# Load the data
data_path = ROOT_DIR / "data/processed/dehar_all_streams_2025.csv"

# Load the csv file
df = pd.read_csv(data_path, index_col="datetime")
df.index = pd.to_datetime(df.index, format="ISO8601", utc=True)
season_df = df.loc["2025-05-01":"2025-10-31"]
print("Data loaded successfully!")
print(f"Data shape: {df.shape}")
print(f"Data columns: {df.columns.tolist()}")

In [ ]:
daily = {}

daily["tair_mean"] = season_df["tair_c"].resample("1D").mean()
daily["tair_std"] = season_df["tair_c"].resample("1D").std()
daily["vpd_mean"] = season_df["vpd_hpa"].resample("1D").mean()
daily["vpd_max"] = season_df["vpd_hpa"].resample("1D").max()
daily["vpd_std"] = season_df["vpd_hpa"].resample("1D").std()
daily['sm_mean'] = season_df["sm_mean_pct"].resample("1D").mean()
daily['sm_std'] = season_df["sm_std_pct"].resample("1D").mean()
# daily['sapflow_mean'] = season_df["js_h10546"].resample("1D").sum(min_count=40)
# daily['sapflow_std'] = season_df["js_h10546"].resample("1D").std()     
# daily["twd_mean"] = season_df["twd_um_h10546"].resample("1D").mean()
# daily["twd_std"]  = season_df["twd_um_h10546"].resample("1D").std()
daily["precip_mm"] = season_df["precip_mm"].resample("1D").sum()

# Trees with all three measurements (SWP + TWD + sap flow)
PHYS_TREES = ["h10545", "h10546", "h10560"]

js_cols = [c for c in season_df.columns if c.startswith("js_h")]
sapflow_daily_all = season_df[js_cols].resample("1D").sum(min_count=40)
daily['sapflow_mean'] = sapflow_daily_all.mean(axis=1)
daily['sapflow_std']  = sapflow_daily_all.std(axis=1)

# TWD: inter-tree mean ± std across matched trees
twd_cols = [f"twd_um_{t}" for t in PHYS_TREES]
twd_daily_all = season_df[twd_cols].resample("1D").mean()
daily["twd_mean"] = twd_daily_all.mean(axis=1)
daily["twd_std"]  = twd_daily_all.std(axis=1)

# Stem water potential
# SWP: inter-tree mean ± std across matched trees (predawn window)
swp_cols = [f"swp_mpa_{t}" for t in PHYS_TREES]

def _sunrise_utc_hour(rg_series: pd.Series, thr_wm2: float = 5.0) -> pd.Series:
    df = rg_series.to_frame("rg")
    df["date"] = df.index.normalize()
    df = df[df["rg"] > thr_wm2]
    first_ts_per_day = df.groupby("date").apply(lambda g: g.index.min())
    # return fractional hour-of-day (UTC)
    return first_ts_per_day.dt.hour + first_ts_per_day.dt.minute / 60.0

sunrise_h = _sunrise_utc_hour(season_df["rg_wm2"])
swp_col   = season_df[f"swp_mpa_{SWP_TREE}"].dropna()

predawn_records = []
for date, sr_h in sunrise_h.items():
    win_end   = pd.Timestamp(date).tz_convert("UTC") + pd.Timedelta(hours=float(sr_h)) \
                if pd.Timestamp(date).tzinfo else pd.Timestamp(date, tz="UTC") + pd.Timedelta(hours=float(sr_h))
    win_start = win_end - pd.Timedelta(hours=2)
    # mean predawn value per tree, then inter-tree stats
    tree_means = []
    for t in PHYS_TREES:
        col = f"swp_mpa_{t}"
        window = season_df[col].dropna().loc[win_start:win_end]
        if len(window):
            tree_means.append(window.mean())
    if tree_means:
        predawn_records.append({
            "date": date,
            "swp_mean": np.mean(tree_means),
            "swp_std":  np.std(tree_means, ddof=1) if len(tree_means) > 1 else 0.0,
            "n": len(tree_means),
        })

swp_pd = pd.DataFrame(predawn_records).set_index("date")
swp_pd.index = pd.DatetimeIndex(swp_pd.index, tz="UTC")
swp_pd["swp_ci95"] = 1.96 * swp_pd["swp_std"] / np.sqrt(swp_pd["n"])
daily["swp_pd"]      = swp_pd["swp_mean"]
daily["swp_pd_ci95"] = swp_pd["swp_ci95"]
daily["swp_std"]    = swp_pd["swp_std"]

# GNSS-T VOD
# Use VOD only when humidity was more than 95% durinig the time of the acquisition
def _filter_vod_by_rh(
    vod_series: pd.Series,
    rh_series: pd.Series,
    precip_series: pd.Series,
    rh_thresh_pct: float,
    rain_thresh_mm: float = 0.001,
    rain_lookback_h: float = 12.0,
) -> pd.Series:
    vod_df = vod_series.to_frame("vod")
    vod_df["rh"] = rh_series.reindex(vod_df.index, method="nearest").values
    
    # Cumulative precip in the N hours before each VOD timestamp
    lookback = pd.Timedelta(hours=rain_lookback_h)
    precip_prior = np.array([
        precip_series.loc[ts - lookback : ts].sum()
        for ts in vod_df.index
    ])
    vod_df["precip_prior"] = precip_prior

    mask = (vod_df["rh"] <= rh_thresh_pct) & (vod_df["precip_prior"] <= rain_thresh_mm)
    return vod_df.loc[mask, "vod"]

RH_THRESH_PCT = 95
# vod_filtered = _filter_vod_by_rh(season_df["nvod_gps1"].dropna(), season_df["rh_pct"].dropna(), season_df["precip_mm"].dropna(), RH_THRESH_PCT)

# # Compute hourly std and identify noisy hours
# vod_hourly_std = vod_filtered.resample("1H").std()

# # Threshold: e.g. mean + 2*std of the hourly-std distribution
# std_thresh = vod_hourly_std.mean() + 2 * vod_hourly_std.std()
# noisy_hours = vod_hourly_std[vod_hourly_std > std_thresh].index

# # Flag observations in those hours as NaN
# vod_filtered_clean = vod_filtered.copy()
# for h in noisy_hours:
#     vod_filtered_clean.loc[h : h + pd.Timedelta(hours=1) - pd.Timedelta(seconds=1)] = np.nan
    
# daily["vod_mean"] = vod_filtered_clean.resample("1D").mean().interpolate(method="time", limit=3)
# daily["vod_std"]  = vod_filtered_clean.resample("1D").std().interpolate(method="time", limit=3)
# GNSS-T VOD — average across all working GPS devices
VOD_COLS = ["nvod_gps1", "nvod_gps3"]

vod_filtered_all = {}
for col in VOD_COLS:
    vod_filtered_all[col] = _filter_vod_by_rh(
        season_df[col].dropna(),
        season_df["rh_pct"].dropna(),
        season_df["precip_mm"].dropna(),
        RH_THRESH_PCT,
    )

# Clean noisy hours per device, then combine
vod_clean_frames = []
for col, vod_filtered in vod_filtered_all.items():
    vod_hourly_std = vod_filtered.resample("1H").std()
    std_thresh = vod_hourly_std.mean() + 5 * vod_hourly_std.std()
    noisy_hours = vod_hourly_std[vod_hourly_std > std_thresh].index
    vod_clean = vod_filtered.copy()
    for h in noisy_hours:
        vod_clean.loc[h : h + pd.Timedelta(hours=1) - pd.Timedelta(seconds=1)] = np.nan
    vod_clean_frames.append(vod_clean.rename(col))

vod_clean_df = pd.concat(vod_clean_frames, axis=1)
vod_daily_all = vod_clean_df.resample("1D").mean()

# daily["vod_mean"] = vod_daily_all.mean(axis=1).interpolate(method="time", limit=3)
# daily["vod_std"]  = vod_daily_all.std(axis=1).interpolate(method="time", limit=3)

# Use only gps1 (single device)
vod_gps1_clean = vod_clean_frames[0]  # nvod_gps1
daily["vod_mean"] = vod_gps1_clean.resample("1D").mean().interpolate(method="time", limit=3)
daily["vod_std"]  = vod_gps1_clean.resample("1D").std().interpolate(method="time", limit=3)


# (vii) Leaf angle — h10546, mean across cam65/66/67, daytime UTC window ────
def _build_leaf_quality_mask(la_idx, precip_series: pd.Series,
                             precip_thr=LEAF_PRECIP_THR_MM, lockout_h=LEAF_PRECIP_LOCKOUT_H):
    mask = pd.Series(True, index=la_idx)
    p = precip_series.reindex(la_idx, method="nearest", tolerance=pd.Timedelta("30min")).fillna(0)
    wet = (p >= precip_thr).astype(float)
    n_steps = max(1, int(np.ceil(lockout_h * 60 / 30)))  # 30-min cadence
    mask &= wet.rolling(n_steps, min_periods=1).max() == 0
    return mask

la_cols = [f"leaf_angle_{c}" for c in LEAF_CAMS if f"leaf_angle_{c}" in season_df.columns]
la_df = season_df[la_cols]

la_mask = _build_leaf_quality_mask(la_df.index, season_df["precip_mm"])
la_filt = la_df[la_mask]

h0, h1 = LEAF_ANGLE_WINDOW_UTC
la_win = la_filt[(la_filt.index.hour >= h0) & (la_filt.index.hour < h1)]

la_daily_mean = la_win.resample("1D").mean()
la_daily_std  = la_win.resample("1D").std()

daily["leaf_angle_cam65"]     = la_daily_mean["leaf_angle_cam65"]
daily["leaf_angle_cam65_std"] = la_daily_std["leaf_angle_cam65"]
daily["leaf_angle_cam67"]     = la_daily_mean["leaf_angle_cam67"]
daily["leaf_angle_cam67_std"] = la_daily_std["leaf_angle_cam67"]

# la_daily_per_cam = la_win.resample("1D").mean()
# daily["leaf_angle_mean"] = la_daily_per_cam.mean(axis=1)
# daily["leaf_angle_std"]  = la_daily_per_cam.std(axis=1)


# PAI — total canopy PAI per scan (cumulative LinearPAI at the top of the profile)
pai_long = pd.read_csv(P["pai"], parse_dates=["datetime"])
pai_long["datetime"] = pd.to_datetime(pai_long["datetime"], utc=True)
pai_long = pai_long[pai_long["datetime"].dt.year == TARGET_YEAR]
pai_long = pai_long[pai_long["quality_good"] == True]   # ← adjust column name if needed

pai = (
    pai_long.groupby("datetime")["HingePAI"].max()
    .to_frame("pai_total")
    .sort_index()
)

daily["pai_total"] = pai["pai_total"]

# (ix) GPP + ET daily mean + REddyProc u* uncertainty ──────────────────────
daily["gpp_mean"] = season_df["gpp_f_umol_m2s"].resample("1D").mean()
daily["gpp_std"]  = season_df["gpp_f_sd_umol_m2s"].resample("1D").mean()

neg_mask = daily["gpp_mean"] < 0
daily["gpp_mean"] = daily["gpp_mean"].mask(neg_mask).interpolate(method="time", limit=5)
daily["gpp_std"]  = daily["gpp_std"].mask(neg_mask).interpolate(method="time", limit=5)

daily["et_mean"]  = season_df["et_f_mm_h"].resample("1D").sum(min_count=1) * 0.5
# et_daytime = season_df["et_f_mm_h"][season_df["et_f_mm_h"] > 0]
# daily["et_std"] = (
#     et_daytime.resample("1D").std() / 
#     et_daytime.resample("1D").count().pow(0.5)
# )


# Phenology - GCC (cam65 & cam67, + all-cam ensemble)
# Quality mask: rain lockout + high ustar (wind proxy)
GCC_PRECIP_THR_MM  = 0.0001   # same as leaf angle
GCC_PRECIP_LOCK_H  = 6.0      # 1 h lockout after any rain step
GCC_USTAR_MAX_MS   = 0.6      # drop timesteps with very high turbulence/wind
GCC_INTERP_LIMIT_D = 5        # max consecutive days to gap-fill

gcc_cols = [c for c in season_df.columns if c.startswith("gcc_cam")]

# Build shared quality mask (same timestamps as season_df)
_p = season_df["precip_mm"].fillna(0)
_wet = (_p >= GCC_PRECIP_THR_MM).astype(float)
_n_lock = max(1, int(np.ceil(GCC_PRECIP_LOCK_H * 60 / 30)))  # steps at 30-min cadence
_rain_mask = _wet.rolling(_n_lock, min_periods=1).max() == 0

_ustar = season_df["ustar_ms"].fillna(0)
_wind_mask = _ustar <= GCC_USTAR_MAX_MS

gcc_qc_mask = _rain_mask & _wind_mask

# Apply mask and compute daily stats per cam
gcc_qc = season_df[gcc_cols].where(gcc_qc_mask)

gcc_cam_daily_mean = gcc_qc.resample("1D").mean()
gcc_cam_daily_p90  = gcc_qc.resample("1D").quantile(0.90)

# Per-cam daily (cam65, cam67)
for cam_id in [65, 67]:
    col = f"gcc_cam{cam_id}"
    daily[f"gcc_cam{cam_id}_mean"] = gcc_cam_daily_mean[col].interpolate(method="time", limit=GCC_INTERP_LIMIT_D)
    daily[f"gcc_cam{cam_id}_p90"]  = gcc_cam_daily_p90[col].interpolate(method="time", limit=GCC_INTERP_LIMIT_D)

# All-cam ensemble: inter-cam mean ± std
daily["gcc_all_mean"]     = gcc_cam_daily_mean.mean(axis=1).interpolate(method="time", limit=GCC_INTERP_LIMIT_D)
daily["gcc_all_mean_std"] = gcc_cam_daily_mean.std(axis=1).interpolate(method="time", limit=GCC_INTERP_LIMIT_D)
daily["gcc_all_p90"]      = gcc_cam_daily_p90.mean(axis=1).interpolate(method="time", limit=GCC_INTERP_LIMIT_D)
daily["gcc_all_p90_std"]  = gcc_cam_daily_p90.std(axis=1).interpolate(method="time", limit=GCC_INTERP_LIMIT_D)

print(f"GCC quality mask: {gcc_qc_mask.sum()} / {len(gcc_qc_mask)} 30-min steps retained "
      f"({gcc_qc_mask.mean()*100:.1f}%)")

In [ ]:
daily["leaf_angle_mean"]
vod_daily = vod_filtered.resample("1D").mean().interpolate(method="time", limit=3)

# Savitzky-Golay: window_length=7 days, polynomial order=3
# window_length must be odd and > polyorder
vod_smooth = pd.Series(
    savgol_filter(vod_daily.fillna(vod_daily.mean()), window_length=11, polyorder=2),
    index=vod_daily.index,
)

vod_daily.plot(label="Filtered VOD", color=TOL_BRIGHT["blue"], alpha=0.4, linewidth=0.8)
vod_smooth.plot(label="SG-smoothed VOD", color=TOL_BRIGHT["blue"], linewidth=1.8)

In [ ]:
vod_smooth = pd.Series(
    savgol_filter(daily["leaf_angle_mean"].fillna(daily["leaf_angle_mean"].mean()), window_length=11, polyorder=2),
    index=daily["leaf_angle_mean"].index,
)

daily["leaf_angle_mean"].plot(label="Filtered VOD", color=TOL_BRIGHT["blue"], alpha=0.4, linewidth=0.8)
vod_smooth.plot(label="SG-smoothed VOD", color=TOL_BRIGHT["blue"], linewidth=1.8)

# Time-series plot

In [ ]:
# Cut the time series to 01.05.2025 - 31.10.2025
daily["pai_total"] = daily["pai_total"].resample("1D").mean()  # ← align to daily index
daily_df = pd.DataFrame(daily)
daily_df = daily_df.loc["2025-05-01":"2025-10-31"]

In [ ]:
from palettable.tableau import Tableau_20

tableau_colors = Tableau_20.hex_colors

fig, ax = plt.subplots(figsize=(10, 1.5))
for i, c in enumerate(tableau_colors):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.text(i + 0.5, -0.3, str(i), ha="center", va="top", fontsize=8)
ax.set_xlim(0, len(tableau_colors))
ax.set_ylim(-0.5, 1)
ax.axis("off")
plt.tight_layout()

In [ ]:
tableau_colors[14]

In [ ]:
from palettable.cartocolors.qualitative import Prism_10
prism_colors = Prism_10.hex_colors

fig, ax = plt.subplots(figsize=(10, 1.5))
for i, c in enumerate(prism_colors):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.text(i + 0.5, -0.3, str(i), ha="center", va="top", fontsize=8)
ax.set_xlim(0, len(prism_colors))
ax.set_ylim(-0.5, 1)
ax.axis("off")
plt.tight_layout()

In [ ]:
colors = ["#a68a59", "#94a659", "#66a659", "#59a67b", "#59a3a6", "#5975a6", "#6b59a6", "#9959a6", "#a65984", "#a65c59"]

fig, ax = plt.subplots(figsize=(10, 1.5))
for i, c in enumerate(colors):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.text(i + 0.5, -0.3, str(i), ha="center", va="top", fontsize=8)
ax.set_xlim(0, len(colors))
ax.set_ylim(-0.5, 1)
ax.axis("off")
plt.tight_layout()

In [ ]:
# Panel 0 
tair_smooth = pd.Series(
    savgol_filter(daily["tair_mean"].fillna(daily["tair_mean"].mean()), window_length=9, polyorder=2),
    index=daily["tair_mean"].index,
)
tair_std_smooth = pd.Series(
    savgol_filter(daily["tair_std"].fillna(daily["tair_std"].mean()), window_length=7, polyorder=2),
    index=daily["tair_std"].index,
)

vpd_smooth = pd.Series(
    savgol_filter(daily["vpd_max"].fillna(daily["vpd_max"].mean()), window_length=7, polyorder=2),
    index=daily["vpd_max"].index,
)

# Panel 1
sm_smooth = pd.Series(
    savgol_filter(daily["sm_mean"].fillna(daily["sm_mean"].mean()), window_length=7, polyorder=2),
    index=daily["sm_mean"].index,
)
sm_std_smooth = pd.Series(
    savgol_filter(daily["sm_std"].fillna(daily["sm_std"].mean()), window_length=5, polyorder=2),
    index=daily["sm_std"].index,
)
precip_smooth = pd.Series(
    savgol_filter(daily["precip_mm"].fillna(daily["precip_mm"].mean()), window_length=3, polyorder=2),
    index=daily["precip_mm"].index,
)

# Panel 2 - Stemwater potential
swp_smooth = pd.Series(
    savgol_filter(daily["swp_pd"].fillna(daily["swp_pd"].mean()), window_length=9, polyorder=2),
    index=daily["swp_pd"].index,
)
swp_std_smooth = pd.Series(
    savgol_filter(daily["swp_std"].fillna(daily["swp_std"].mean()), window_length=5, polyorder=2),
    index=daily["swp_std"].index,
)


# Panel 3 - Sap flow and tree water deficit
sapflow_smooth = pd.Series(
    savgol_filter(daily["sapflow_mean"].fillna(daily["sapflow_mean"].mean()), window_length=9, polyorder=2),
    index=daily["sapflow_mean"].index,
)
sapflow_std_smooth = pd.Series(
    savgol_filter(daily["sapflow_std"].fillna(daily["sapflow_std"].mean()), window_length=5, polyorder=2),
    index=daily["sapflow_std"].index,
)  
twd_smooth = pd.Series(
    savgol_filter(daily["twd_mean"].fillna(daily["twd_mean"].mean()), window_length=9, polyorder=2),
    index=daily["twd_mean"].index,
)
twd_std_smooth = pd.Series(
    savgol_filter(daily["twd_std"].fillna(daily["twd_std"].mean()), window_length=5, polyorder=2),
    index=daily["twd_std"].index,
)


# Panel 4 - GNSS VOD
vod_smooth = pd.Series(
    savgol_filter(daily["vod_mean"].fillna(daily["vod_mean"].mean()), window_length=7, polyorder=2),
    index=daily["vod_mean"].index,
)
vod_std_smooth = pd.Series(
    savgol_filter(daily["vod_std"].fillna(daily["vod_std"].mean()), window_length=5, polyorder=2),
    index=daily["vod_std"].index,
)

# Panel 5 - Leaf angle
# Panel 5 - Leaf angle (per camera)
la65_smooth = pd.Series(
    savgol_filter(daily_df["leaf_angle_cam65"].fillna(daily_df["leaf_angle_cam65"].mean()), window_length=7, polyorder=2),
    index=daily_df["leaf_angle_cam65"].index,
)
la65_std_smooth = pd.Series(
    savgol_filter(daily_df["leaf_angle_cam65_std"].fillna(daily_df["leaf_angle_cam65_std"].mean()), window_length=5, polyorder=2),
    index=daily_df["leaf_angle_cam65_std"].index,
)
la67_smooth = pd.Series(
    savgol_filter(daily_df["leaf_angle_cam67"].fillna(daily_df["leaf_angle_cam67"].mean()), window_length=7, polyorder=2),
    index=daily_df["leaf_angle_cam67"].index,
)
la67_std_smooth = pd.Series(
    savgol_filter(daily_df["leaf_angle_cam67_std"].fillna(daily_df["leaf_angle_cam67_std"].mean()), window_length=5, polyorder=2),
    index=daily_df["leaf_angle_cam67_std"].index,
)

# Panel 6 - PAI
_pai_filled = daily_df["pai_total"].interpolate(method="time", limit=30).fillna(daily_df["pai_total"].median())
pai_smooth = pd.Series(
    savgol_filter(_pai_filled, window_length=7, polyorder=2),
    index=daily_df["pai_total"].index,
)
# Clip smooth line to actual data range (first to last valid scan)
pai_smooth = pai_smooth.loc[
    daily_df["pai_total"].first_valid_index():
    daily_df["pai_total"].last_valid_index()
]

# Panel 6 - GCC p90 (all-cam ensemble)
_gcc_p90_filled = daily_df["gcc_all_p90"].interpolate(method="time", limit=5).fillna(daily_df["gcc_all_p90"].mean())
gcc_p90_smooth = pd.Series(
    savgol_filter(_gcc_p90_filled, window_length=7, polyorder=2),
    index=daily_df["gcc_all_p90"].index,
)
gcc_p90_std_smooth = pd.Series(
    savgol_filter(
        daily_df["gcc_all_p90_std"].fillna(daily_df["gcc_all_p90_std"].mean()),
        window_length=5, polyorder=2,
    ),
    index=daily_df["gcc_all_p90_std"].index,
)

# Panel 7 - GPP and ET
gpp_smooth = pd.Series(
    savgol_filter(daily["gpp_mean"].fillna(daily["gpp_mean"].mean()), window_length=7, polyorder=2),
    index=daily["gpp_mean"].index,
)
gpp_std_smooth = pd.Series(
    savgol_filter(daily["gpp_std"].fillna(daily["gpp_std"].mean()), window_length=5, polyorder=2),
    index=daily["gpp_std"].index,
)
et_smooth = pd.Series(
    savgol_filter(daily["et_mean"].fillna(daily["et_mean"].mean()), window_length=7, polyorder=2),
    index=daily["et_mean"].index,
)

In [ ]:
# ── Figure parameters ────────────────────────────────────────────────────────
FIG_SIZE          = (16, 22)
LINE_WIDTH        = 3.25
FILL_ALPHA        = 0.25         # default uncertainty band alpha
FILL_ALPHA_SWP    = 0.25         # SWP band (slightly more opaque)

YLABEL_FONTSIZE   = 20          # y-axis label font size
TICK_LABELSIZE    = 18           # tick label font size
LEGEND_FONTSIZE   = 13           # legend font size

GRID_COLOR        = "#e6e6e6"
GRID_LW           = 1.5

# X-axis limits
X_MIN = pd.Timestamp("2025-05-01", tz="UTC") - pd.Timedelta(days=1)
X_MAX = pd.Timestamp("2025-10-31", tz="UTC") + pd.Timedelta(days=1)

# Stress shading
STRESS_PERIODS = [
    (pd.Timestamp("2025-06-16", tz="UTC"), pd.Timestamp("2025-07-03", tz="UTC")),
    (pd.Timestamp("2025-08-06", tz="UTC"), pd.Timestamp("2025-08-20", tz="UTC")),
]
STRESS_ALPHA = 0.2

# Panel-specific y-axis scaling
VPD_YLIM_MULT      = 1.75   # vpd_smooth.max() * this → upper ylim
VPD_YLIM_LOWER     = -2
PRECIP_YLIM_MULT   = 1.75   # inverted precip axis upper bound
TWD_YLIM_UPPER_M   = 1.75   # twd_smooth.max() * this → (inverted) upper bound
TWD_YLIM_LOWER_M   = 0.25   # twd_smooth.max() * this → lower negative buffer
ET_YLIM_MULT       = 1.35

# Panel 1 precipitation bar
PRECIP_BAR_WIDTH   = 1
PRECIP_BAR_ALPHA   = 0.5

# Panel 5 leaf angle second line color
LA_LOW_COLOR       = "#708f7e"

# Panel 6 GCC color and PAI scatter
GCC_COLOR          = "#73886f"
GCC_FILL_ALPHA     = 0.25
PAI_SCATTER_ALPHA  = 0.5
PAI_SCATTER_SIZE   = 20

# ── Helper: apply common styling to a primary axis ───────────────────────────
def _style_ax(ax, color, ylabel, ytick_size=TICK_LABELSIZE, ylabel_size=YLABEL_FONTSIZE):
    ax.set_ylabel(ylabel, color=color, fontsize=ylabel_size)
    ax.spines["left"].set_color(color)
    ax.tick_params(axis="y", labelcolor=color, color=color, labelsize=ytick_size)
    ax.grid(axis="x", linestyle="--", linewidth=GRID_LW, color=GRID_COLOR)
    ax.grid(axis="y", linestyle="", linewidth=0)

def _style_twin(ax, color, ylabel, ytick_size=TICK_LABELSIZE, ylabel_size=YLABEL_FONTSIZE):
    ax.set_ylabel(ylabel, color=color, fontsize=ylabel_size)
    ax.spines["right"].set_color(color)
    ax.tick_params(axis="y", labelcolor=color, color=color, labelsize=ytick_size)
    ax.grid(False)

# ── Build figure ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(8, 1, figsize=FIG_SIZE)

# ------ Panel 0: Tair + VPD ------
axes[0].plot(tair_smooth, color=colors[9], linewidth=LINE_WIDTH)
axes[0].fill_between(tair_std_smooth.index,
    tair_smooth - tair_std_smooth, tair_smooth + tair_std_smooth,
    color=colors[9], alpha=FILL_ALPHA, linewidth=0)
_style_ax(axes[0], colors[9], r"$\mathrm{T_{air}}$ (°C)")

ax0_twin = axes[0].twinx()
ax0_twin.plot(vpd_smooth, color=tableau_colors[14], linewidth=LINE_WIDTH)
ax0_twin.set_ylim(VPD_YLIM_LOWER, vpd_smooth.max() * VPD_YLIM_MULT)
_style_twin(ax0_twin, tableau_colors[14], r"$\mathrm{VPD}_{\max}$ (hPa)")

# ------ Panel 1: Soil moisture + Precipitation ------
axes[1].plot(sm_smooth, color=colors[8], linewidth=LINE_WIDTH)
axes[1].fill_between(sm_std_smooth.index,
    sm_smooth - sm_std_smooth, sm_smooth + sm_std_smooth,
    color=colors[8], alpha=FILL_ALPHA, linewidth=0)
_style_ax(axes[1], colors[8], "VWC (%)")

ax1_twin = axes[1].twinx()
ax1_twin.bar(daily["precip_mm"].index, daily["precip_mm"].values,
             color=tableau_colors[0], alpha=PRECIP_BAR_ALPHA,
             width=PRECIP_BAR_WIDTH, edgecolor="none")
ax1_twin.set_ylim(daily["precip_mm"].max() * PRECIP_YLIM_MULT, 0)
_style_twin(ax1_twin, tableau_colors[0], "Precipitation (mm)")

# ------ Panel 2: Stem water potential ------
axes[2].plot(swp_smooth, color=colors[7], linewidth=LINE_WIDTH)
axes[2].fill_between(swp_std_smooth.index,
    swp_smooth - swp_std_smooth, swp_smooth + swp_std_smooth,
    color=colors[7], alpha=FILL_ALPHA_SWP, linewidth=0)
_style_ax(axes[2], colors[7], r"$\Psi_\mathrm{Stem}$ (MPa)")

# ------ Panel 3: Sap flow + TWD ------
axes[3].plot(sapflow_smooth, color=colors[5], linewidth=LINE_WIDTH)
axes[3].fill_between(sapflow_std_smooth.index,
    sapflow_smooth - sapflow_std_smooth, sapflow_smooth + sapflow_std_smooth,
    color=colors[5], alpha=FILL_ALPHA, linewidth=0)
_style_ax(axes[3], colors[5], r"$\mathrm{J_s}$ (cm³ cm⁻² d⁻¹)")

ax3_twin = axes[3].twinx()
ax3_twin.plot(twd_smooth, color=colors[6], linewidth=LINE_WIDTH)
ax3_twin.fill_between(twd_std_smooth.index,
    twd_smooth - twd_std_smooth, twd_smooth + twd_std_smooth,
    color=colors[6], alpha=FILL_ALPHA, linewidth=0)
ax3_twin.set_ylim(twd_smooth.max() * TWD_YLIM_UPPER_M, -twd_smooth.max() * TWD_YLIM_LOWER_M)
_style_twin(ax3_twin, colors[6], "TWD (µm)")

# ------ Panel 4: GNSS VOD ------
axes[4].plot(vod_smooth, color=colors[4], linewidth=LINE_WIDTH)
axes[4].fill_between(vod_smooth.index,
    vod_smooth - vod_std_smooth, vod_smooth + vod_std_smooth,
    color=colors[4], alpha=FILL_ALPHA, linewidth=0)
_style_ax(axes[4], colors[4], "VOD (–)")

# ------ Panel 5: Leaf angle ------
axes[5].plot(la67_smooth, label="8.5 m", color=colors[3], linewidth=LINE_WIDTH)
axes[5].fill_between(la67_std_smooth.index,
    la67_smooth - la67_std_smooth, la67_smooth + la67_std_smooth,
    color=colors[3], alpha=FILL_ALPHA, linewidth=0)
axes[5].plot(la65_smooth, label="3.5 m", color=LA_LOW_COLOR, linewidth=LINE_WIDTH, linestyle="--")
axes[5].fill_between(la65_std_smooth.index,
    la65_smooth - la65_std_smooth, la65_smooth + la65_std_smooth,
    color=LA_LOW_COLOR, alpha=FILL_ALPHA, linewidth=0)
axes[5].legend(fontsize=LEGEND_FONTSIZE)
_style_ax(axes[5], colors[3], "Leaf angle (°)")

# ------ Panel 6: PAI + GCC ------
axes[6].plot(pai_smooth, color=colors[2], linewidth=LINE_WIDTH)
# axes[6].scatter(daily_df["pai_total"].index, daily_df["pai_total"].values,
#     color=colors[2], alpha=PAI_SCATTER_ALPHA, s=PAI_SCATTER_SIZE, zorder=5)
_style_ax(axes[6], colors[2], "PAI (–)")

ax6_twin = axes[6].twinx()
ax6_twin.plot(gcc_p90_smooth, color=GCC_COLOR, linewidth=LINE_WIDTH)
ax6_twin.fill_between(gcc_p90_std_smooth.index,
    gcc_p90_smooth - gcc_p90_std_smooth, gcc_p90_smooth + gcc_p90_std_smooth,
    color=GCC_COLOR, alpha=GCC_FILL_ALPHA, linewidth=0)
_style_twin(ax6_twin, GCC_COLOR, "GCC (–)")

# ------ Panel 7: GPP + ET ------
axes[7].plot(gpp_smooth, color=colors[1], linewidth=LINE_WIDTH)
axes[7].fill_between(gpp_std_smooth.index,
    gpp_smooth - gpp_std_smooth, gpp_smooth + gpp_std_smooth,
    color=colors[1], alpha=FILL_ALPHA, linewidth=0)
_style_ax(axes[7], colors[1], "GPP (µmol m⁻² s⁻¹)")

ax7_twin = axes[7].twinx()
ax7_twin.plot(et_smooth, color=colors[0], linewidth=LINE_WIDTH)
ax7_twin.set_ylim(0, et_smooth.max() * ET_YLIM_MULT)
_style_twin(ax7_twin, colors[0], "ET (mm d⁻¹)")

# ── Stress shading, x-limits, x-tick labels ──────────────────────────────────
for i, _ax in enumerate(axes):
    _ax.set_xlim(X_MIN, X_MAX)
    for t_start, t_end in STRESS_PERIODS:
        _ax.axvspan(t_start, t_end,
                    color=STRESS_SHADE_COLOR, alpha=STRESS_ALPHA, zorder=0, linewidth=0)
    if i < len(axes) - 1:
        _ax.set_xticklabels([])
        _ax.tick_params(axis="x", bottom=False)
    else:
        _ax.tick_params(axis="x", labelsize=TICK_LABELSIZE)
        _ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        _ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.tight_layout()
plt.show()

# ── Export ───────────────────────────────────────────────────────────────────
out_path = FIG_DIR / "timeseries_poster.pdf"
fig.savefig(out_path, dpi=300, bbox_inches="tight", format="pdf")
print(f"Saved → {out_path}")

## 2. Daily data preparation

In [ ]:
from scipy.signal import savgol_filter as _sg_filt

SG_WINDOW     = 7
SG_ORDER      = 2
SEASON_START  = "2025-05-01"
SEASON_END    = "2025-10-31"
EXPORT_PATH   = DATA_DIR / "processed" / "dehar_daily_season_2025_filtered.csv"

daily_idx = pd.date_range(SEASON_START, SEASON_END, freq="1D", tz="UTC")

def _smooth_like_plot(
    s: pd.Series,
    interp_limit: int | None = None,
    fallback: str = "mean",          # "mean" | "median"
    clip_to_valid_range: bool = False
) -> pd.Series:
    s = s.reindex(daily_idx)

    # Match plot behavior per stream
    if interp_limit is not None:
        filled = s.interpolate(method="time", limit=interp_limit)
    else:
        filled = s.copy()

    if fallback == "median":
        filled = filled.fillna(s.median())
    else:
        filled = filled.fillna(s.mean())

    sg = pd.Series(
        _sg_filt(filled.values, window_length=SG_WINDOW, polyorder=SG_ORDER),
        index=s.index
    )

    # Optional: match PAI plot clipping (only between first/last real observation)
    if clip_to_valid_range and s.notna().any():
        first_valid = s.first_valid_index()
        last_valid  = s.last_valid_index()
        sg = sg.where((sg.index >= first_valid) & (sg.index <= last_valid))

    return sg

def _add(
    df: pd.DataFrame,
    series: pd.Series,
    col_raw: str,
    col_sg: str,
    interp_limit: int | None = None,
    fallback: str = "mean",
    clip_to_valid_range: bool = False
):
    s = series.reindex(daily_idx)
    df[col_raw] = s.values
    df[col_sg] = _smooth_like_plot(
        s,
        interp_limit=interp_limit,
        fallback=fallback,
        clip_to_valid_range=clip_to_valid_range
    ).values

out = pd.DataFrame(index=daily_idx)
out.index.name = "date"

# Meteorology
_add(out, daily["tair_mean"],   "tair_c_mean",            "tair_c_mean_sg")
_add(out, daily["tair_std"],    "tair_c_std",             "tair_c_std_sg")
_add(out, daily["vpd_max"],     "vpd_hpa_max",            "vpd_hpa_max_sg")
_add(out, daily["precip_mm"],   "precip_mm",              "precip_mm_sg")

# Soil moisture
_add(out, daily["sm_mean"],     "sm_pct_mean",            "sm_pct_mean_sg")
_add(out, daily["sm_std"],      "sm_pct_std",             "sm_pct_std_sg")

# Stem water potential
_add(out, daily["swp_pd"],      "swp_mpa_predawn_mean",   "swp_mpa_predawn_mean_sg")
_add(out, daily["swp_std"],     "swp_mpa_predawn_std",    "swp_mpa_predawn_std_sg")

# Sap flow
_add(out, daily["sapflow_mean"],"sapflow_jscm3cm2d_mean", "sapflow_jscm3cm2d_mean_sg")
_add(out, daily["sapflow_std"], "sapflow_jscm3cm2d_std",  "sapflow_jscm3cm2d_std_sg")

# TWD
_add(out, daily["twd_mean"],    "twd_um_mean",            "twd_um_mean_sg")
_add(out, daily["twd_std"],     "twd_um_std",             "twd_um_std_sg")

# VOD
_add(out, daily["vod_mean"],    "vod_mean",               "vod_mean_sg")
_add(out, daily["vod_std"],     "vod_std",                "vod_std_sg")

# Leaf angle
_add(out, daily["leaf_angle_cam65"],     "leaf_angle_cam65_mean", "leaf_angle_cam65_mean_sg")
_add(out, daily["leaf_angle_cam65_std"], "leaf_angle_cam65_std",  "leaf_angle_cam65_std_sg")
_add(out, daily["leaf_angle_cam67"],     "leaf_angle_cam67_mean", "leaf_angle_cam67_mean_sg")
_add(out, daily["leaf_angle_cam67_std"], "leaf_angle_cam67_std",  "leaf_angle_cam67_std_sg")

# PAI: EXACTLY like plotted version
_add(
    out, daily["pai_total"], "pai_total", "pai_total_sg",
    interp_limit=30, fallback="median", clip_to_valid_range=True
)

# GCC p90: like plotted version
_add(
    out, daily["gcc_all_p90"], "gcc_p90_mean", "gcc_p90_mean_sg",
    interp_limit=5, fallback="mean"
)
_add(out, daily["gcc_all_p90_std"], "gcc_p90_std", "gcc_p90_std_sg")

# GPP + ET
_add(out, daily["gpp_mean"], "gpp_umolm2s_mean", "gpp_umolm2s_mean_sg")
_add(out, daily["gpp_std"],  "gpp_umolm2s_std",  "gpp_umolm2s_std_sg")
_add(out, daily["et_mean"],  "et_mmd_mean",      "et_mmd_mean_sg")

# Sentinel-1
s1 = pd.read_csv(
    DATA_DIR / "processed/satellite/sentinel1_filtered_2025.csv",
    index_col="time", parse_dates=True
)
s1.index = pd.DatetimeIndex(s1.index, tz="UTC")
s1 = s1.loc[SEASON_START:SEASON_END].reindex(daily_idx)
for col in s1.columns:
    out[f"s1_{col.lower()}"] = s1[col].values

# Sentinel-2
s2 = pd.read_csv(
    DATA_DIR / "processed/satellite/sentinel2_filtered_2025.csv",
    index_col="time", parse_dates=True
)
s2.index = pd.DatetimeIndex(s2.index, tz="UTC")
s2 = s2.drop(columns=["interpolated"], errors="ignore")
s2 = s2.loc[SEASON_START:SEASON_END].reindex(daily_idx)
for col in s2.columns:
    out[f"s2_{col.lower()}"] = s2[col].values

out.to_csv(EXPORT_PATH)
print(f"Exported {out.shape[0]} rows x {out.shape[1]} columns")
print(f"-> {EXPORT_PATH}")

# Analysis

## Cross-correlation lag analysis (event-centered)

This section computes lagged cross-correlation on Savitzky-Golay columns from the exported daily dataframe (`out` CSV).

Design choices:
- Event-centered windows around the August stress event, defined as first downward crossing of SWP predawn SG below -1 MPa.
- Fixed sample size across lags (no edge-effect shrinkage): each lag uses the same number of points by symmetric trimming.
- Optional first-difference preprocessing to reduce autocorrelation influence.
- Output includes pairwise lag table and a network-style lead-lag plot.

In [ ]:
# Setup: load SG dataframe, detect event date, define windows/parameters
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

EXPORT_PATH = DATA_DIR / "processed" / "dehar_daily_season_2025_filtered.csv"
out_cc = pd.read_csv(EXPORT_PATH, index_col="date", parse_dates=True)
out_cc.index = pd.to_datetime(out_cc.index, utc=True)

# Use only Savitzky-Golay columns
sg_cols = [c for c in out_cc.columns if c.endswith("_sg")]
print(f"Loaded: {EXPORT_PATH}")
print(f"Rows: {len(out_cc)}, SG columns: {len(sg_cols)}")

# Analysis settings
WINDOW_HALFWIDTHS = [10, 15, 20, 30]      # days around event
MAX_LAG = 10                               # days; keep within event-local dynamics
USE_FIRST_DIFFERENCE = True                # helps reduce autocorrelation effects
MIN_ABS_R_FOR_NETWORK = 0.45               # edge strength threshold for visualization
TOP_EDGES_MAX = 45                         # avoid unreadable network

SWP_COL = "swp_mpa_predawn_mean_sg"
SWP_CROSS_THRESHOLD = -1.0

def find_august_swp_crossing(s: pd.Series, threshold: float = -1.0) -> pd.Timestamp:
    """First downward crossing in August; fallback to August nearest threshold."""
    s = s.sort_index()
    aug = s[s.index.month == 8].dropna()
    if aug.empty:
        raise ValueError("No August SWP values available for event detection.")

    downward = aug[(aug.shift(1) > threshold) & (aug <= threshold)]
    if not downward.empty:
        return downward.index[0]

    # Fallback if no strict crossing exists
    return (aug - threshold).abs().idxmin()

event_date = find_august_swp_crossing(out_cc[SWP_COL], threshold=SWP_CROSS_THRESHOLD)
print(f"Event date (SWP crossing ~ {SWP_CROSS_THRESHOLD:.1f} MPa): {event_date.date()}")
print(f"Windows (±days): {WINDOW_HALFWIDTHS} | Max lag: ±{MAX_LAG} d | First diff: {USE_FIRST_DIFFERENCE}")

In [ ]:
# Compute fixed-window lagged cross-correlation for each event window
# Definition used here:
# - The reference window is fixed on var_i (SWP-centered event window dates).
# - For each lag, var_j is sampled at shifted dates (t + lag).
# - n is constant across lags for a given window (no shrinking with lag).
# - Correlations are only accepted when n >= MIN_FIXED_N_FOR_CORR.

MIN_FIXED_N_FOR_CORR = 30

def _prep_series_full(s: pd.Series, use_first_difference: bool) -> pd.Series:
    s = s.copy()
    if use_first_difference:
        s = s.diff()
    # Fill on the full timeline so shifted lookups keep fixed n per window
    s = s.interpolate(method="time", limit=5).ffill().bfill()
    return s

def fixed_window_lag_corr(
    s_ref_full: pd.Series,
    s_shift_full: pd.Series,
    ref_index: pd.DatetimeIndex,
    max_lag: int,
) -> pd.DataFrame:
    """
    Corr between ref(t) and shift(t+lag) on a fixed reference index.
    Positive lag means reference variable leads shifted variable.
    n is constant for all tested lags inside one window.
    """
    n_fixed = len(ref_index)
    if n_fixed < MIN_FIXED_N_FOR_CORR:
        return pd.DataFrame(columns=["lag", "r", "n", "max_lag_used"])

    t_min = s_shift_full.index.min()
    t_max = s_shift_full.index.max()
    lag_min_feasible = (t_min - ref_index[0]).days
    lag_max_feasible = (t_max - ref_index[-1]).days

    lag_low = max(-max_lag, lag_min_feasible)
    lag_high = min(max_lag, lag_max_feasible)
    if lag_low > lag_high:
        return pd.DataFrame(columns=["lag", "r", "n", "max_lag_used"])

    rows = []
    ref_vals = s_ref_full.reindex(ref_index).to_numpy(dtype=float)
    max_lag_used = int(max(abs(lag_low), abs(lag_high)))

    for lag in range(int(lag_low), int(lag_high) + 1):
        shifted_idx = ref_index + pd.Timedelta(days=int(lag))
        shift_vals = s_shift_full.reindex(shifted_idx).to_numpy(dtype=float)

        # Keep strict fixed-n behavior: skip lag if any missing value remains
        if np.isnan(ref_vals).any() or np.isnan(shift_vals).any():
            continue

        r = np.corrcoef(ref_vals, shift_vals)[0, 1]
        rows.append({
            "lag": int(lag),
            "r": float(r),
            "n": int(n_fixed),
            "max_lag_used": int(max_lag_used),
        })

    return pd.DataFrame(rows)

# Precompute full preprocessed series once
prep_full = pd.DataFrame(index=out_cc.index)
for c in sg_cols:
    prep_full[c] = _prep_series_full(out_cc[c], use_first_difference=USE_FIRST_DIFFERENCE)

results_by_window = {}
pair_rows = []

PAIR_COLS = [
    "window_halfwidth_days",
    "event_date",
    "var_i",
    "var_j",
    "best_lag_days",
    "best_r",
    "abs_best_r",
    "n_fixed",
    "max_lag_used",
    "window_start",
    "window_end",
]

for half_w in WINDOW_HALFWIDTHS:
    w0 = event_date - pd.Timedelta(days=half_w)
    w1 = event_date + pd.Timedelta(days=half_w)

    ref_index = pd.date_range(w0, w1, freq="1D", tz="UTC")
    n_ref = len(ref_index)
    if n_ref < MIN_FIXED_N_FOR_CORR:
        print(f"Window ±{half_w:2d} d: skipped (n={n_ref} < {MIN_FIXED_N_FOR_CORR})")
        win_pairs_df = pd.DataFrame(columns=PAIR_COLS)
        results_by_window[half_w] = win_pairs_df
        pair_rows.append(win_pairs_df)
        continue

    window_pairs = []
    for i, ci in enumerate(sg_cols):
        for j in range(i + 1, len(sg_cols)):
            cj = sg_cols[j]
            ccf = fixed_window_lag_corr(
                prep_full[ci], prep_full[cj], ref_index=ref_index, max_lag=MAX_LAG
            )
            if ccf.empty:
                continue

            best_idx = ccf["r"].abs().idxmax()
            best = ccf.loc[best_idx]
            window_pairs.append({
                "window_halfwidth_days": half_w,
                "event_date": event_date.date().isoformat(),
                "var_i": ci,
                "var_j": cj,
                "best_lag_days": int(best["lag"]),
                "best_r": float(best["r"]),
                "abs_best_r": float(abs(best["r"])),
                "n_fixed": int(best["n"]),
                "max_lag_used": int(best["max_lag_used"]),
                "window_start": w0.date().isoformat(),
                "window_end": w1.date().isoformat(),
            })

    win_pairs_df = pd.DataFrame(window_pairs, columns=PAIR_COLS).sort_values(
        "abs_best_r", ascending=False
    )
    results_by_window[half_w] = win_pairs_df
    pair_rows.append(win_pairs_df)
    print(f"Window ±{half_w:2d} d: pairs={len(win_pairs_df)} | fixed n={n_ref}")

lag_summary = pd.concat(pair_rows, ignore_index=True) if pair_rows else pd.DataFrame(columns=PAIR_COLS)

# Save outputs for later use
LAG_SUMMARY = lag_summary
LAG_RESULTS_BY_WINDOW = results_by_window

display_cols = [
    "window_halfwidth_days", "var_i", "var_j", "best_lag_days", "best_r",
    "abs_best_r", "n_fixed", "max_lag_used"
]
print("\nTop lag relationships (by |r|):")
display(LAG_SUMMARY[display_cols].head(30))

In [ ]:
# Network-style visualization of lead-lag structure
def _pretty_name(c: str) -> str:
    return c.replace("_sg", "")

def build_lead_edges(df_pairs: pd.DataFrame, min_abs_r: float = 0.45) -> pd.DataFrame:
    """Convert pairwise lag result to directed leader->follower edges."""
    edges = []
    for _, r in df_pairs.iterrows():
        if abs(r["best_r"]) < min_abs_r:
            continue
        lag = int(r["best_lag_days"])
        if lag == 0:
            continue

        if lag > 0:
            # var_i leads var_j
            src, dst = r["var_i"], r["var_j"]
            lag_abs = lag
        else:
            # negative lag: var_j leads var_i
            src, dst = r["var_j"], r["var_i"]
            lag_abs = -lag

        edges.append({
            "source": src,
            "target": dst,
            "lag_days": lag_abs,
            "r": float(r["best_r"]),
            "abs_r": float(abs(r["best_r"])),
            "n_fixed": int(r["n_fixed"]),
        })

    if not edges:
        return pd.DataFrame(columns=["source", "target", "lag_days", "r", "abs_r", "n_fixed"])
    out_e = pd.DataFrame(edges).sort_values("abs_r", ascending=False)
    return out_e.head(TOP_EDGES_MAX)

def draw_lead_lag_network(edges: pd.DataFrame, title: str):
    if edges.empty:
        print("No edges above threshold. Lower MIN_ABS_R_FOR_NETWORK to visualize more links.")
        return

    nodes = sorted(set(edges["source"]).union(set(edges["target"])))

    # Lead score: outgoing abs_r minus incoming abs_r
    score = {n: 0.0 for n in nodes}
    for _, e in edges.iterrows():
        score[e["source"]] += e["abs_r"]
        score[e["target"]] -= e["abs_r"]

    # Left->right layout: leaders on left, followers on right
    ordered = sorted(nodes, key=lambda n: score[n], reverse=True)
    n_nodes = len(ordered)
    x_left, x_right = -1.45, 1.45
    x_vals = np.linspace(x_left, x_right, n_nodes) if n_nodes > 1 else np.array([0.0])

    # Vertical spacing with slight alternation to reduce overlap
    y_base = np.linspace(1.0, -1.0, n_nodes) if n_nodes > 1 else np.array([0.0])
    pos = {}
    for i, n in enumerate(ordered):
        jitter = 0.10 if i % 2 == 0 else -0.10
        pos[n] = (float(x_vals[i]), float(y_base[i] + jitter))

    fig, ax = plt.subplots(figsize=(14, 7))

    # Draw edges and lag labels
    for _, e in edges.iterrows():
        x1, y1 = pos[e["source"]]
        x2, y2 = pos[e["target"]]
        lw = 1.0 + 4.0 * e["abs_r"]
        alpha = 0.35 + 0.60 * min(1.0, e["abs_r"])
        color = "#2a9d8f" if e["r"] >= 0 else "#e76f51"
        rad = 0.12 if (y2 - y1) >= 0 else -0.12
        arrow = FancyArrowPatch(
            (x1, y1), (x2, y2),
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>",
            mutation_scale=12 + 1.0 * e["lag_days"],
            linewidth=lw, color=color, alpha=alpha,
            shrinkA=18, shrinkB=18,
        )
        ax.add_patch(arrow)

        # Lag-day label near mid-edge
        mx, my = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        dy = 0.05 if rad > 0 else -0.05
        ax.text(
            mx, my + dy, f"{int(e['lag_days'])} d",
            fontsize=8, color="#1f2937", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75),
            zorder=4,
        )

    # Draw nodes and labels
    for n in ordered:
        x, y = pos[n]
        node_color = "#1d3557" if n == SWP_FOCUS_NODE else "#264653"
        node_size = 520 if n == SWP_FOCUS_NODE else 420
        ax.scatter(x, y, s=node_size, c=node_color, edgecolors="white", linewidths=1.3, zorder=5)
        ax.text(x, y + 0.16, _pretty_name(n), ha="center", va="bottom", fontsize=9)

    ax.set_title(title, fontsize=14)
    ax.set_xlim(-1.7, 1.7)
    ax.set_ylim(-1.25, 1.25)
    ax.axis("off")

    # Side guides for interpretation
    ax.text(-1.62, 1.15, "Leading variables", fontsize=10, fontweight="bold", ha="left")
    ax.text(1.62, 1.15, "Following variables", fontsize=10, fontweight="bold", ha="right")
    ax.plot([-1.58, -1.20], [1.09, 1.09], color="#264653", lw=2)
    ax.plot([1.20, 1.58], [1.09, 1.09], color="#264653", lw=2)
    plt.show()

# Pick a window to visualize (change here quickly)
WINDOW_TO_PLOT = 20
SWP_FOCUS_NODE = "swp_mpa_predawn_mean_sg"

pairs_plot = LAG_RESULTS_BY_WINDOW.get(WINDOW_TO_PLOT, pd.DataFrame())
edges_all = build_lead_edges(pairs_plot, min_abs_r=MIN_ABS_R_FOR_NETWORK)

# Keep only SWP-centered links (incoming or outgoing)
edges_plot = edges_all[
    (edges_all["source"] == SWP_FOCUS_NODE) | (edges_all["target"] == SWP_FOCUS_NODE)
].copy()
edges_plot = edges_plot.sort_values("abs_r", ascending=False).head(TOP_EDGES_MAX)

print(f"Window ±{WINDOW_TO_PLOT} d | SWP-centered edges shown: {len(edges_plot)}")
display(edges_plot.head(20))

draw_lead_lag_network(
    edges_plot,
    title=f"SWP-centered lead-lag network ({event_date.date()}), window ±{WINDOW_TO_PLOT} d"
),

# Optional export of lag table
lag_csv = DATA_DIR / "processed" / "cross_correlation_lags_event_windows_2025.csv"
LAG_SUMMARY.to_csv(lag_csv, index=False)
print(f"Saved lag summary -> {lag_csv}")

## Worked example: SWP predawn vs leaf angle cam67 (lag cross-correlation)

This example shows exactly how lagged cross-correlation is computed for one variable pair:
- `swp_mpa_predawn_mean_sg`
- `leaf_angle_cam67_mean_sg`

It uses a fixed number of points for all tested lags (constant `n`) and plots:
1. correlation vs lag with optimal lag marked,
2. scatter plots for several lags including the optimum.

In [ ]:
# Example pair lag analysis: SWP predawn SG vs leaf angle cam67 SG
# This example uses STRICT fixed-window n across lags.
PAIR_X = "swp_mpa_predawn_mean_sg"
PAIR_Y = "leaf_angle_cam67_mean_sg"
# PAIR_X = "leaf_angle_cam67_mean_sg"
# PAIR_Y = "swp_mpa_predawn_mean_sg"
EXAMPLE_WINDOW_DAYS = 10         # gives 31 points (2*15+1)
EXAMPLE_MAX_LAG = 30
EXAMPLE_MIN_FIXED_N = 20
EXAMPLE_USE_FIRST_DIFFERENCE = False  # keep SG values directly for interpretation

# Ensure dataframe exists
if "out_cc" not in globals():
    out_cc = pd.read_csv(EXPORT_PATH, index_col="date", parse_dates=True)
    out_cc.index = pd.to_datetime(out_cc.index, utc=True)

if "event_date" not in globals():
    event_date = find_august_swp_crossing(out_cc[SWP_COL], threshold=SWP_CROSS_THRESHOLD)

# Fixed reference window (never changes with lag)
w0 = event_date - pd.Timedelta(days=EXAMPLE_WINDOW_DAYS)
w1 = event_date + pd.Timedelta(days=EXAMPLE_WINDOW_DAYS)
ref_index = pd.date_range(w0, w1, freq="1D", tz="UTC")
n_fixed = len(ref_index)

if n_fixed < EXAMPLE_MIN_FIXED_N:
    raise ValueError(
        f"Fixed window has n={n_fixed}, below EXAMPLE_MIN_FIXED_N={EXAMPLE_MIN_FIXED_N}. "
        "Increase EXAMPLE_WINDOW_DAYS."
    )

sx = out_cc[PAIR_X].copy()
sy = out_cc[PAIR_Y].copy()
if EXAMPLE_USE_FIRST_DIFFERENCE:
    sx = sx.diff()
    sy = sy.diff()

# Fill on full timeline, then sample shifted values on fixed reference dates
sx = sx.interpolate(method="time", limit=5).ffill().bfill()
sy = sy.interpolate(method="time", limit=5).ffill().bfill()

# Feasible lag bounds so shifted timestamps stay in available data range
t_min = sy.index.min()
t_max = sy.index.max()
lag_min_feasible = (t_min - ref_index[0]).days
lag_max_feasible = (t_max - ref_index[-1]).days
lag_low = max(-EXAMPLE_MAX_LAG, lag_min_feasible)
lag_high = min(EXAMPLE_MAX_LAG, lag_max_feasible)
if lag_low > lag_high:
    raise ValueError("No feasible lags with current window and data bounds.")

x_ref = sx.reindex(ref_index).to_numpy(dtype=float)
if np.isnan(x_ref).any():
    raise ValueError("Reference SWP window contains missing values after fill.")

rows = []
for lag in range(int(lag_low), int(lag_high) + 1):
    y_shift = sy.reindex(ref_index + pd.Timedelta(days=int(lag))).to_numpy(dtype=float)
    if np.isnan(y_shift).any():
        continue
    r = np.corrcoef(x_ref, y_shift)[0, 1]
    rows.append({"lag": int(lag), "r": float(r), "n": int(n_fixed)})

ccf_df = pd.DataFrame(rows)
if ccf_df.empty:
    raise ValueError("No valid lag correlations could be computed.")

best_idx = ccf_df["r"].abs().idxmax()
best = ccf_df.loc[best_idx]
best_lag = int(best["lag"])
best_r = float(best["r"])

lead_text = (
    f"{PAIR_X} leads {PAIR_Y} by {best_lag} d" if best_lag > 0 else
    (f"{PAIR_Y} leads {PAIR_X} by {-best_lag} d" if best_lag < 0 else "No lead/lag (0 d)")
)

print(f"Fixed window: {w0.date()} to {w1.date()} | fixed n = {n_fixed}")
print(f"Tested lags: {int(lag_low)}..{int(lag_high)} d")
print(f"Best lag (by |r|): {best_lag} d | r = {best_r:.3f}")
print(lead_text)

# Plot 1: correlation vs lag
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(ccf_df["lag"], ccf_df["r"], marker="o", lw=1.8, color="#264653")
ax.axhline(0, color="#999999", lw=1.0, ls="--")
ax.axvline(best_lag, color="#E76F51", lw=1.6, ls="--", label=f"opt lag = {best_lag} d")
ax.scatter([best_lag], [best_r], color="#E76F51", s=70, zorder=3)
ax.set_xlabel("Lag (days), positive = SWP leads leaf angle")
ax.set_ylabel("Pearson r")
ax.set_title("SWP predawn vs leaf angle cam67: lag cross-correlation")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Plot 2: scatter at selected lags (same fixed n for all panels)
lag_candidates = [int(lag_low), 0, int(lag_high), best_lag]
lags_to_plot = []
for l in lag_candidates:
    if l not in lags_to_plot:
        lags_to_plot.append(int(l))

fig, axes = plt.subplots(1, len(lags_to_plot), figsize=(4.8 * len(lags_to_plot), 4.2), sharey=True)
if len(lags_to_plot) == 1:
    axes = [axes]

for ax, lag in zip(axes, lags_to_plot):
    y_shift = sy.reindex(ref_index + pd.Timedelta(days=int(lag))).to_numpy(dtype=float)
    r_lag = np.corrcoef(x_ref, y_shift)[0, 1]

    ax.scatter(x_ref, y_shift, s=28, alpha=0.75, color="#2a9d8f", edgecolor="none")
    if len(x_ref) > 1:
        m, b = np.polyfit(x_ref, y_shift, 1)
        xx = np.linspace(np.min(x_ref), np.max(x_ref), 60)
        ax.plot(xx, m * xx + b, color="#1f2937", lw=1.4)

    ax.set_title(f"lag {lag} d | r={r_lag:.2f} | n={n_fixed}")
    ax.set_xlabel("SWP predawn SG")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Leaf angle cam67 SG (shifted)")
plt.suptitle("Scatter comparison across lags (strict fixed window n)", y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

## Lag robustness diagnostics (window + direction sensitivity)

This diagnostic checks one variable pair across multiple event windows and both direction conventions:
- `x -> y`: fixed reference on `x`, shift `y`
- `y -> x`: fixed reference on `y`, shift `x`

Outputs:
1. Window-by-window lag table with peak and near-peak plateau width.
2. Consensus lag by direction (median, IQR).
3. Block-bootstrap confidence interval of the best lag for a selected primary direction/window.
4. Stability flag (`stable`/`unstable`) based on consistency criteria.

In [ ]:
out_cc.columns

In [ ]:
[i for i in range(15, 31, 2)]

In [ ]:
# Robust lag diagnostics for one pair across windows and both directions

# ---------------- User settings ----------------
STAB_PAIR_X = "twd_um_mean" #"swp_mpa_predawn_mean_sg"  #"swp_mpa_predawn_mean_sg"
STAB_PAIR_Y = "swp_mpa_predawn_mean" #"vod_mean_sg" #"leaf_angle_cam67_mean_sg" #"swp_mpa_predawn_mean_sg" #"leaf_angle_cam67_mean_sg"
STAB_WINDOWS = [i for i in range(10, 26, 1)]  # half-widths in days
STAB_MAX_LAG = 15
STAB_MIN_FIXED_N = 10
STAB_USE_FIRST_DIFFERENCE = False
PLATEAU_FRAC = 0.95              # near-peak definition: |r| >= 95% of peak |r|

# Primary setup for bootstrap CI on best lag
PRIMARY_DIRECTION = "x->y"      # "x->y" or "y->x"
PRIMARY_WINDOW = 20
BOOT_N = 1000
BOOT_BLOCK_LEN = 3
BOOT_SEED = 42

# Stability thresholds
STABLE_MIN_WINDOWS = 3
STABLE_MAX_IQR_DAYS = 4
STABLE_MIN_SIGN_CONSISTENCY = 0.80


def _prep_stab(s: pd.Series, use_first_difference: bool) -> pd.Series:
    s = s.copy()
    if use_first_difference:
        s = s.diff()
    return s.interpolate(method="time", limit=5).ffill().bfill()


def _lag_curve_fixed(ref_series: pd.Series, shift_series: pd.Series,
                     ref_index: pd.DatetimeIndex, max_lag: int) -> pd.DataFrame:
    n_fixed = len(ref_index)
    if n_fixed < STAB_MIN_FIXED_N:
        return pd.DataFrame(columns=["lag", "r", "n"])

    t_min = shift_series.index.min()
    t_max = shift_series.index.max()
    lag_min_feasible = (t_min - ref_index[0]).days
    lag_max_feasible = (t_max - ref_index[-1]).days

    lag_low = max(-max_lag, lag_min_feasible)
    lag_high = min(max_lag, lag_max_feasible)
    if lag_low > lag_high:
        return pd.DataFrame(columns=["lag", "r", "n"])

    x_ref = ref_series.reindex(ref_index).to_numpy(dtype=float)
    if np.isnan(x_ref).any():
        return pd.DataFrame(columns=["lag", "r", "n"])

    rows = []
    for lag in range(int(lag_low), int(lag_high) + 1):
        y_shift = shift_series.reindex(ref_index + pd.Timedelta(days=int(lag))).to_numpy(dtype=float)
        if np.isnan(y_shift).any():
            continue
        r = np.corrcoef(x_ref, y_shift)[0, 1]
        rows.append({"lag": int(lag), "r": float(r), "n": int(n_fixed)})
    return pd.DataFrame(rows)


def _summarize_curve(ccf_df: pd.DataFrame, plateau_frac: float) -> dict:
    best_idx = ccf_df["r"].abs().idxmax()
    best = ccf_df.loc[best_idx]
    peak_abs = float(abs(best["r"]))
    cut = peak_abs * float(plateau_frac)
    plateau = ccf_df[ccf_df["r"].abs() >= cut]

    return {
        "best_lag": int(best["lag"]),
        "best_r": float(best["r"]),
        "peak_abs_r": peak_abs,
        "plateau_min_lag": int(plateau["lag"].min()),
        "plateau_max_lag": int(plateau["lag"].max()),
        "plateau_width_days": int(plateau["lag"].max() - plateau["lag"].min()),
    }


def _sign_consistency(lags: pd.Series) -> float:
    nz = lags[lags != 0]
    if len(nz) == 0:
        return 1.0
    pos = (nz > 0).mean()
    neg = (nz < 0).mean()
    return float(max(pos, neg))


def _classify_stability(direction_df: pd.DataFrame) -> dict:
    if direction_df.empty:
        return {
            "n_windows": 0,
            "consensus_lag_median": np.nan,
            "lag_q25": np.nan,
            "lag_q75": np.nan,
            "lag_iqr": np.nan,
            "sign_consistency": np.nan,
            "median_abs_r": np.nan,
            "stability_flag": "insufficient-data",
        }

    lags = direction_df["best_lag_days"]
    q25 = float(np.percentile(lags, 25))
    q75 = float(np.percentile(lags, 75))
    iqr = q75 - q25
    sign_cons = _sign_consistency(lags)
    med_abs_r = float(direction_df["peak_abs_r"].median())

    stable = (
        len(direction_df) >= STABLE_MIN_WINDOWS
        and iqr <= STABLE_MAX_IQR_DAYS
        and sign_cons >= STABLE_MIN_SIGN_CONSISTENCY
    )

    return {
        "n_windows": int(len(direction_df)),
        "consensus_lag_median": float(np.median(lags)),
        "lag_q25": q25,
        "lag_q75": q75,
        "lag_iqr": float(iqr),
        "sign_consistency": float(sign_cons),
        "median_abs_r": med_abs_r,
        "stability_flag": "stable" if stable else "unstable",
    }


def _build_bootstrap_indices(n: int, block_len: int, rng: np.random.Generator) -> np.ndarray:
    idx = []
    while len(idx) < n:
        start = int(rng.integers(0, max(1, n - block_len + 1)))
        idx.extend(range(start, min(start + block_len, n)))
    return np.array(idx[:n], dtype=int)


def _bootstrap_best_lag(ref_series: pd.Series, shift_series: pd.Series,
                        ref_index: pd.DatetimeIndex, lag_low: int, lag_high: int,
                        n_boot: int, block_len: int, seed: int) -> pd.DataFrame:
    x_ref = ref_series.reindex(ref_index).to_numpy(dtype=float)

    lag_grid = list(range(int(lag_low), int(lag_high) + 1))
    y_by_lag = {}
    for lag in lag_grid:
        yv = shift_series.reindex(ref_index + pd.Timedelta(days=int(lag))).to_numpy(dtype=float)
        if np.isnan(yv).any():
            return pd.DataFrame(columns=["boot", "best_lag", "best_r"])
        y_by_lag[lag] = yv

    if np.isnan(x_ref).any():
        return pd.DataFrame(columns=["boot", "best_lag", "best_r"])

    n = len(x_ref)
    rng = np.random.default_rng(seed)
    rows = []
    for b in range(n_boot):
        ii = _build_bootstrap_indices(n=n, block_len=block_len, rng=rng)
        best_lag = None
        best_r = None
        best_abs = -np.inf
        for lag in lag_grid:
            r = float(np.corrcoef(x_ref[ii], y_by_lag[lag][ii])[0, 1])
            if abs(r) > best_abs:
                best_abs = abs(r)
                best_lag = lag
                best_r = r
        rows.append({"boot": b, "best_lag": int(best_lag), "best_r": float(best_r)})
    return pd.DataFrame(rows)


# Ensure source data/event are available
if "out_cc" not in globals():
    out_cc = pd.read_csv(EXPORT_PATH, index_col="date", parse_dates=True)
    out_cc.index = pd.to_datetime(out_cc.index, utc=True)

if "event_date" not in globals():
    event_date = find_august_swp_crossing(out_cc[SWP_COL], threshold=SWP_CROSS_THRESHOLD)

sx_full = _prep_stab(out_cc[STAB_PAIR_X], STAB_USE_FIRST_DIFFERENCE)
sy_full = _prep_stab(out_cc[STAB_PAIR_Y], STAB_USE_FIRST_DIFFERENCE)

print("Lead/lag convention:")
print(f"x->y: lag > 0 means {STAB_PAIR_X} leads {STAB_PAIR_Y}; lag < 0 means {STAB_PAIR_Y} leads {STAB_PAIR_X}.")
print(f"y->x: lag > 0 means {STAB_PAIR_Y} leads {STAB_PAIR_X}; lag < 0 means {STAB_PAIR_X} leads {STAB_PAIR_Y}.")
print("Correlation sign (r) shows in-phase (+) or inverse (-), not who leads.")

rows = []
curves = {}
for half_w in STAB_WINDOWS:
    w0 = event_date - pd.Timedelta(days=half_w)
    w1 = event_date + pd.Timedelta(days=half_w)
    ref_index = pd.date_range(w0, w1, freq="1D", tz="UTC")
    n_fixed = len(ref_index)

    for direction, ref_series, shift_series in [
        ("x->y", sx_full, sy_full),
        ("y->x", sy_full, sx_full),
    ]:
        key = (direction, half_w)

        if n_fixed < STAB_MIN_FIXED_N:
            rows.append({
                "direction": direction,
                "window_halfwidth_days": int(half_w),
                "n_fixed": int(n_fixed),
                "status": f"skip_n<{STAB_MIN_FIXED_N}",
            })
            continue

        ccf_df = _lag_curve_fixed(ref_series, shift_series, ref_index, STAB_MAX_LAG)
        if ccf_df.empty:
            rows.append({
                "direction": direction,
                "window_halfwidth_days": int(half_w),
                "n_fixed": int(n_fixed),
                "status": "skip_no_feasible_lags",
            })
            continue

        s = _summarize_curve(ccf_df, plateau_frac=PLATEAU_FRAC)
        rows.append({
            "direction": direction,
            "window_halfwidth_days": int(half_w),
            "n_fixed": int(n_fixed),
            "status": "ok",
            "best_lag_days": s["best_lag"],
            "best_r": s["best_r"],
            "peak_abs_r": s["peak_abs_r"],
            "plateau_min_lag": s["plateau_min_lag"],
            "plateau_max_lag": s["plateau_max_lag"],
            "plateau_width_days": s["plateau_width_days"],
        })
        curves[key] = {
            "ccf_df": ccf_df,
            "ref_index": ref_index,
            "ref_series": ref_series,
            "shift_series": shift_series,
        }

stab_df = pd.DataFrame(rows)
STAB_RESULTS = stab_df.copy()

print(f"Event date: {event_date.date()} | pair: {STAB_PAIR_X} vs {STAB_PAIR_Y}")
print(f"Windows: {STAB_WINDOWS} | max lag: ±{STAB_MAX_LAG} d | min n: {STAB_MIN_FIXED_N}")
print("\nPer-window lag results:")
display(stab_df)

# Consensus summary by direction
ok_df = stab_df[stab_df["status"] == "ok"].copy()
summary_rows = []
for direction in ["x->y", "y->x"]:
    ddf = ok_df[ok_df["direction"] == direction]
    s = _classify_stability(ddf)
    s["direction"] = direction
    summary_rows.append(s)

summary_df = pd.DataFrame(summary_rows)[[
    "direction", "n_windows", "consensus_lag_median", "lag_q25", "lag_q75", "lag_iqr",
    "sign_consistency", "median_abs_r", "stability_flag"
]]

STAB_SUMMARY = summary_df.copy()
print("\nConsensus by direction:")
display(summary_df)

# Combined median across BOTH plotted directions, converted to x->y sign convention
# x->y lag is used directly; y->x lag is sign-flipped so all values are comparable.
if not ok_df.empty:
    combined_df = ok_df.copy()
    combined_df["lag_x_to_y_equiv"] = np.where(
        combined_df["direction"] == "x->y",
        combined_df["best_lag_days"],
        -combined_df["best_lag_days"],
    )

    combined_median = float(np.median(combined_df["lag_x_to_y_equiv"]))
    combined_median_day = int(np.rint(combined_median))

    COMBINED_MEDIAN_TABLE = combined_df[[
        "direction", "window_halfwidth_days", "best_lag_days", "lag_x_to_y_equiv"
    ]].sort_values(["window_halfwidth_days", "direction"])
    COMBINED_MEDIAN_LAG_X_TO_Y = combined_median
    COMBINED_MEDIAN_LAG_DAY_X_TO_Y = combined_median_day

    if combined_median_day > 0:
        combined_text = f"{STAB_PAIR_X} leads {STAB_PAIR_Y} by ~{combined_median_day} day(s)"
    elif combined_median_day < 0:
        combined_text = f"{STAB_PAIR_Y} leads {STAB_PAIR_X} by ~{abs(combined_median_day)} day(s)"
    else:
        combined_text = f"No lead/lag (about 0 days) between {STAB_PAIR_X} and {STAB_PAIR_Y}"

    print("\nCombined median (both directions, canonical x->y sign):")
    print(f"median lag = {combined_median:.2f} d  (rounded: {combined_median_day} d)")
    print(f"Interpretation: {combined_text}")
    display(COMBINED_MEDIAN_TABLE)
else:
    COMBINED_MEDIAN_TABLE = pd.DataFrame()
    COMBINED_MEDIAN_LAG_X_TO_Y = np.nan
    COMBINED_MEDIAN_LAG_DAY_X_TO_Y = np.nan
    print("\nCombined median skipped: no valid lag rows.")

# Bootstrap CI for selected primary setup
boot_key = (PRIMARY_DIRECTION, PRIMARY_WINDOW)
if boot_key not in curves:
    print(
        f"\nBootstrap skipped: primary setup {PRIMARY_DIRECTION}, ±{PRIMARY_WINDOW} d not available."
    )
    BOOT_RESULTS = pd.DataFrame()
else:
    c = curves[boot_key]
    ccf = c["ccf_df"]
    lag_low = int(ccf["lag"].min())
    lag_high = int(ccf["lag"].max())

    boot_df = _bootstrap_best_lag(
        ref_series=c["ref_series"],
        shift_series=c["shift_series"],
        ref_index=c["ref_index"],
        lag_low=lag_low,
        lag_high=lag_high,
        n_boot=BOOT_N,
        block_len=BOOT_BLOCK_LEN,
        seed=BOOT_SEED,
    )
    BOOT_RESULTS = boot_df.copy()

    if boot_df.empty:
        print("\nBootstrap skipped: missing values in primary setup after alignment.")
    else:
        ci_low = float(np.percentile(boot_df["best_lag"], 2.5))
        ci_high = float(np.percentile(boot_df["best_lag"], 97.5))
        med_boot = float(np.median(boot_df["best_lag"]))

        print(
            f"\nBootstrap lag CI ({PRIMARY_DIRECTION}, window ±{PRIMARY_WINDOW} d): "
            f"median={med_boot:.1f} d, 95% CI [{ci_low:.1f}, {ci_high:.1f}] d "
            f"(n_boot={BOOT_N}, block={BOOT_BLOCK_LEN})"
        )

# Quick diagnostic plot: best lag vs window for both directions
if not ok_df.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    for direction, marker, color in [
        ("x->y", "o", "#1d3557"),
        ("y->x", "s", "#e76f51"),
    ]:
        ddf = ok_df[ok_df["direction"] == direction].sort_values("window_halfwidth_days")
        if ddf.empty:
            continue
        ax.plot(
            ddf["window_halfwidth_days"],
            ddf["best_lag_days"],
            marker=marker,
            linewidth=1.8,
            color=color,
            label=direction,
        )

    ax.axhline(0, color="#777777", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Window half-width (days)")
    ax.set_ylabel("Best lag (days)")
    ax.set_title("Lag sensitivity across window size and direction")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## All-pairs lead/follow analysis (SG only, no std)

This section scales the robust lag workflow to all SG variables (excluding any standard-deviation streams), computes one leader/follower decision per pair from a canonical median lag, quantifies uncertainty via bootstrap over window/max-lag setups, and builds tables plus a poster-oriented network layout.

In [ ]:
# Full recomputation: all-pairs lead/follow inference with setup-bootstrap uncertainty
from itertools import combinations
from matplotlib.patches import FancyArrowPatch

# ---------------- User settings ----------------
ALLPAIR_WINDOWS = [i for i in range(6, 15, 1)]
ALLPAIR_MAXLAGS = [10, 15, 20]
ALLPAIR_MIN_FIXED_N = 10
ALLPAIR_USE_FIRST_DIFFERENCE = False
ALLPAIR_BOOT_N = 500
ALLPAIR_BOOT_SEED = 42
ALLPAIR_MIN_VALID_SETUPS = 12
ALLPAIR_EDGE_TOP_N = 160

# Include satellite indices in all-pairs analysis
ALLPAIR_INCLUDE_SATELLITE = True
ALLPAIR_SATELLITE_SMOOTH_ONLY = True  # keep only *_savgol (and any *_sg if present)
ALLPAIR_SAT_PREFIXES = ("s1_", "s2_")

# Explicit user-requested exclusions
ALLPAIR_EXCLUDE_EXACT = {
    "s1_vv_savgol",
    "s1_vh_savgol",
}
ALLPAIR_EXCLUDE_BASE = {
    "precip_mm",
    "sm_pct_mean",
}

def _is_sg_nonstd(col: str) -> bool:
    c = col.lower()
    return c.endswith("_sg") and ("std" not in c)

def _is_satellite_candidate(col: str, smooth_only: bool = True) -> bool:
    c = col.lower()
    is_sat = any(c.startswith(p) for p in ALLPAIR_SAT_PREFIXES)
    if (not is_sat) or ("std" in c):
        return False
    if not smooth_only:
        return True
    return ("savgol" in c) or c.endswith("_sg")

def _is_excluded_variable(col: str) -> bool:
    c = col.lower()
    if c in ALLPAIR_EXCLUDE_EXACT:
        return True
    for base in ALLPAIR_EXCLUDE_BASE:
        if c == base or c.startswith(base + "_"):
            return True
    return False

def _select_allpair_columns(df: pd.DataFrame) -> tuple[list[str], pd.DataFrame, list[str]]:
    sg_nonstd = sorted([c for c in df.columns if _is_sg_nonstd(c)])

    sat = []
    if ALLPAIR_INCLUDE_SATELLITE:
        sat = sorted([
            c for c in df.columns
            if _is_satellite_candidate(c, smooth_only=ALLPAIR_SATELLITE_SMOOTH_ONLY)
        ])

    selected = sorted(dict.fromkeys(sg_nonstd + sat))
    selected = [c for c in selected if not _is_excluded_variable(c)]

    excluded_present = sorted([c for c in df.columns if _is_excluded_variable(c)])

    meta_rows = []
    for c in selected:
        c_low = c.lower()
        if any(c_low.startswith(p) for p in ALLPAIR_SAT_PREFIXES):
            family = "satellite"
        else:
            family = "sg_nonstd"
        meta_rows.append({"column": c, "family": family})
    meta_df = pd.DataFrame(meta_rows)

    return selected, meta_df, excluded_present

def _prep_allpair_series(s: pd.Series, use_first_difference: bool, daily_index: pd.DatetimeIndex) -> pd.Series:
    s = s.reindex(daily_index).astype(float)
    if use_first_difference:
        s = s.diff()
    return s.interpolate(method="time").ffill().bfill()

def _lag_curve_fixed_ap(ref_series: pd.Series, shift_series: pd.Series,
                        ref_index: pd.DatetimeIndex, max_lag: int, min_n: int) -> pd.DataFrame:
    n_fixed = len(ref_index)
    if n_fixed < min_n:
        return pd.DataFrame(columns=["lag", "r", "n"])

    t_min = shift_series.index.min()
    t_max = shift_series.index.max()
    lag_min_feasible = (t_min - ref_index[0]).days
    lag_max_feasible = (t_max - ref_index[-1]).days

    lag_low = max(-max_lag, lag_min_feasible)
    lag_high = min(max_lag, lag_max_feasible)
    if lag_low > lag_high:
        return pd.DataFrame(columns=["lag", "r", "n"])

    x_ref = ref_series.reindex(ref_index).to_numpy(dtype=float)
    if np.isnan(x_ref).any():
        return pd.DataFrame(columns=["lag", "r", "n"])

    rows = []
    for lag in range(int(lag_low), int(lag_high) + 1):
        y_shift = shift_series.reindex(ref_index + pd.Timedelta(days=int(lag))).to_numpy(dtype=float)
        if np.isnan(y_shift).any():
            continue
        r = np.corrcoef(x_ref, y_shift)[0, 1]
        rows.append({"lag": int(lag), "r": float(r), "n": int(n_fixed)})
    return pd.DataFrame(rows)

def _pair_bootstrap_median_lag(lags: np.ndarray, n_boot: int, rng: np.random.Generator) -> np.ndarray:
    n = len(lags)
    out = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        ii = rng.integers(0, n, size=n)
        out[b] = float(np.median(lags[ii]))
    return out

def _pair_leader_follower(x: str, y: str, median_lag: float) -> tuple[str, str]:
    if median_lag > 0:
        return x, y
    if median_lag < 0:
        return y, x
    return (x, y) if x <= y else (y, x)

if "out_cc" not in globals():
    out_cc = pd.read_csv(EXPORT_PATH, index_col="date", parse_dates=True)
    out_cc.index = pd.to_datetime(out_cc.index, utc=True)

if "event_date" not in globals():
    event_date = find_august_swp_crossing(out_cc[SWP_COL], threshold=SWP_CROSS_THRESHOLD)

daily_idx_all = pd.date_range(
    out_cc.index.min().normalize(),
    out_cc.index.max().normalize(),
    freq="1D",
    tz="UTC",
)

allpair_cols, prep_meta, excluded_present = _select_allpair_columns(out_cc)
if not allpair_cols:
    raise ValueError("No eligible variables found for all-pairs analysis.")

prep_all = pd.DataFrame(index=daily_idx_all)
for c in allpair_cols:
    prep_all[c] = _prep_allpair_series(out_cc[c], ALLPAIR_USE_FIRST_DIFFERENCE, daily_idx_all)

n_sat = int((prep_meta["family"] == "satellite").sum()) if not prep_meta.empty else 0
n_sg = int((prep_meta["family"] == "sg_nonstd").sum()) if not prep_meta.empty else 0

print(f"Event date: {event_date.date()}")
print(f"Eligible variables: {len(allpair_cols)} (sg_nonstd={n_sg}, satellite={n_sat})")
print(f"Excluded by request: {len(excluded_present)}")
if excluded_present:
    print(", ".join(excluded_present))
print(f"Windows={ALLPAIR_WINDOWS} | MaxLags={ALLPAIR_MAXLAGS} | min n={ALLPAIR_MIN_FIXED_N}")
print(f"Pairs to evaluate: {len(allpair_cols) * (len(allpair_cols) - 1) // 2}")

setup_rows = []
pair_rows = []

for pidx, (x_col, y_col) in enumerate(combinations(allpair_cols, 2), start=1):
    sx = prep_all[x_col]
    sy = prep_all[y_col]

    for half_w in ALLPAIR_WINDOWS:
        w0 = event_date - pd.Timedelta(days=int(half_w))
        w1 = event_date + pd.Timedelta(days=int(half_w))
        ref_index = pd.date_range(w0, w1, freq="1D", tz="UTC")
        n_fixed = len(ref_index)

        for max_lag in ALLPAIR_MAXLAGS:
            for direction, ref_s, shift_s, sign_factor in [
                ("x->y", sx, sy, 1),
                ("y->x", sy, sx, -1),
            ]:
                ccf = _lag_curve_fixed_ap(
                    ref_series=ref_s,
                    shift_series=shift_s,
                    ref_index=ref_index,
                    max_lag=int(max_lag),
                    min_n=int(ALLPAIR_MIN_FIXED_N),
                )
                if ccf.empty:
                    continue

                best = ccf.loc[ccf["r"].abs().idxmax()]
                lag_raw = int(best["lag"])
                lag_xy_equiv = int(sign_factor * lag_raw)

                setup_rows.append({
                    "var_x": x_col,
                    "var_y": y_col,
                    "direction": direction,
                    "window_halfwidth_days": int(half_w),
                    "max_lag_days": int(max_lag),
                    "n_fixed": int(n_fixed),
                    "best_lag_days": lag_raw,
                    "lag_x_to_y_equiv": lag_xy_equiv,
                    "best_r": float(best["r"]),
                    "peak_abs_r": float(abs(best["r"])),
                    "best_r2": float(best["r"] ** 2),
                })

    setup_df_pair = pd.DataFrame([r for r in setup_rows if r["var_x"] == x_col and r["var_y"] == y_col])
    n_setups = int(len(setup_df_pair))
    if n_setups == 0:
        continue

    lag_vals = setup_df_pair["lag_x_to_y_equiv"].to_numpy(dtype=float)
    median_lag = float(np.median(lag_vals))
    median_lag_day = int(np.rint(median_lag))

    seed_pair = ALLPAIR_BOOT_SEED + (hash((x_col, y_col)) % 1000000)
    rng = np.random.default_rng(seed_pair)
    boot_medians = _pair_bootstrap_median_lag(lag_vals, n_boot=int(ALLPAIR_BOOT_N), rng=rng)

    ci_low = float(np.percentile(boot_medians, 2.5))
    ci_high = float(np.percentile(boot_medians, 97.5))
    iqr_boot = float(np.percentile(boot_medians, 75) - np.percentile(boot_medians, 25))
    sign_consistency = float(max((lag_vals > 0).mean(), (lag_vals < 0).mean(), (lag_vals == 0).mean()))

    leader, follower = _pair_leader_follower(x_col, y_col, median_lag)

    pair_rows.append({
        "var_x": x_col,
        "var_y": y_col,
        "leader": leader,
        "follower": follower,
        "median_lag_x_to_y": median_lag,
        "median_lag_x_to_y_day": median_lag_day,
        "median_abs_r": float(setup_df_pair["peak_abs_r"].median()),
        "median_r2": float(setup_df_pair["best_r2"].median()),
        "median_r_signed": float(setup_df_pair["best_r"].median()),
        "n_valid_setups": n_setups,
        "bootstrap_iqr_days": iqr_boot,
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "bootstrap_ci_width": float(ci_high - ci_low),
        "sign_consistency": sign_consistency,
        "uncertainty_score_raw": iqr_boot,
    })

    if pidx % 25 == 0:
        print(f"Processed {pidx} pairs...")

pair_setup_df = pd.DataFrame(setup_rows)
pair_summary_df = pd.DataFrame(pair_rows)
if pair_summary_df.empty:
    raise ValueError("No pairwise lag results were produced.")

pair_summary_df = pair_summary_df[pair_summary_df["n_valid_setups"] >= int(ALLPAIR_MIN_VALID_SETUPS)].copy()
if pair_summary_df.empty:
    raise ValueError("All pairs filtered out by ALLPAIR_MIN_VALID_SETUPS.")

u = pair_summary_df["uncertainty_score_raw"].to_numpy(dtype=float)
u_lo = float(np.quantile(u, 0.05))
u_hi = float(np.quantile(u, 0.95))
if np.isclose(u_hi, u_lo):
    u_norm = np.zeros_like(u)
else:
    u_clipped = np.clip(u, u_lo, u_hi)
    u_norm = (u_clipped - u_lo) / (u_hi - u_lo)

pair_summary_df["uncertainty_norm"] = u_norm
pair_summary_df["certainty_norm"] = 1.0 - pair_summary_df["uncertainty_norm"]
pair_summary_df["edge_width"] = (0.8 + 4.2 * pair_summary_df["certainty_norm"]) * pair_summary_df["median_r2"]
pair_summary_df["edge_alpha"] = 0.25 + 0.65 * pair_summary_df["certainty_norm"]

vars_all = sorted(set(pair_summary_df["var_x"]).union(set(pair_summary_df["var_y"])))
lead_counts = pair_summary_df["leader"].value_counts()
follow_counts = pair_summary_df["follower"].value_counts()

var_rows = []
for v in vars_all:
    n_lead = int(lead_counts.get(v, 0))
    n_follow = int(follow_counts.get(v, 0))
    n_total = n_lead + n_follow
    lead_fraction = float(n_lead / n_total) if n_total > 0 else np.nan
    follow_fraction = float(n_follow / n_total) if n_total > 0 else np.nan
    lead_tendency = float((n_lead - n_follow) / n_total) if n_total > 0 else np.nan

    involved = pair_summary_df[(pair_summary_df["leader"] == v) | (pair_summary_df["follower"] == v)]
    var_unc = float(involved["uncertainty_norm"].median()) if not involved.empty else np.nan

    var_rows.append({
        "variable": v,
        "lead_count": n_lead,
        "follow_count": n_follow,
        "n_pairs_involved": n_total,
        "lead_fraction": lead_fraction,
        "follow_fraction": follow_fraction,
        "lead_tendency": lead_tendency,
        "uncertainty_norm": var_unc,
    })

var_summary_df = pd.DataFrame(var_rows).sort_values(["lead_tendency", "uncertainty_norm"], ascending=[False, True])

node_df = var_summary_df.copy()
node_df["x"] = -node_df["lead_tendency"]
node_df["y"] = node_df["uncertainty_norm"]

edge_df = pair_summary_df.copy()
edge_df["source"] = edge_df["leader"]
edge_df["target"] = edge_df["follower"]
edge_df = edge_df.sort_values(["certainty_norm", "median_abs_r"], ascending=[False, False]).head(int(ALLPAIR_EDGE_TOP_N))

plot_nodes = node_df.set_index("variable")
fig, ax = plt.subplots(figsize=(13, 7))
for _, e in edge_df.iterrows():
    s = e["source"]
    t = e["target"]
    if s not in plot_nodes.index or t not in plot_nodes.index:
        continue
    x1, y1 = float(plot_nodes.loc[s, "x"]), float(plot_nodes.loc[s, "y"])
    x2, y2 = float(plot_nodes.loc[t, "x"]), float(plot_nodes.loc[t, "y"])
    rad = 0.10 if (y2 - y1) >= 0 else -0.10
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle="-|>",
        mutation_scale=9 + 5 * float(e["certainty_norm"]),
        linewidth=float(e["edge_width"]),
        color="#2a9d8f" if float(e["median_lag_x_to_y"]) * float(e["median_abs_r"]) >= 0 else "#e76f51",
        alpha=float(e["edge_alpha"]),
        shrinkA=12,
        shrinkB=12,
    )
    ax.add_patch(arrow)

sc = ax.scatter(
    node_df["x"], node_df["y"],
    s=180 + 260 * node_df["n_pairs_involved"] / max(1, node_df["n_pairs_involved"].max()),
    c=node_df["lead_tendency"],
    cmap="coolwarm_r",
    edgecolors="white",
    linewidths=1.2,
    zorder=3,
)

for _, r in node_df.iterrows():
    ax.text(float(r["x"]), float(r["y"]) + 0.025, str(r["variable"]).replace("_sg", ""), ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Leading tendency (left = more leading, right = more following)")
ax.set_ylabel("Uncertainty (top = higher)")
ax.set_title("All-variable lead/follow map with bootstrap uncertainty")
ax.grid(alpha=0.25)
ax.axvline(0, color="#777777", linestyle="--", linewidth=1.0)
ax.set_ylim(-0.02, 1.05)
cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Lead tendency score")
plt.tight_layout()
out_path = FIG_DIR / "correlation_network.pdf"
fig.savefig(out_path, dpi=300, bbox_inches="tight", format="pdf")
plt.show()

ALLPAIR_COLUMNS = allpair_cols
ALLPAIR_INPUT_DAILY = prep_all.copy()
ALLPAIR_PREP_METADATA = prep_meta.copy()
PAIR_SETUP_RESULTS = pair_setup_df.copy()
PAIR_LEADFOLLOW_TABLE = pair_summary_df.copy()
VARIABLE_LEADFOLLOW_TABLE = var_summary_df.copy()
NETWORK_NODE_TABLE = node_df.copy()
NETWORK_EDGE_TABLE = edge_df.copy()

pair_csv = DATA_DIR / "processed" / "allpairs_lead_follow_uncertainty_2025.csv"
var_csv = DATA_DIR / "processed" / "variable_lead_follow_counts_2025.csv"
node_csv = DATA_DIR / "processed" / "network_nodes_lead_uncertainty_2025.csv"
edge_csv = DATA_DIR / "processed" / "network_edges_lead_uncertainty_2025.csv"
PAIR_LEADFOLLOW_TABLE.to_csv(pair_csv, index=False)
VARIABLE_LEADFOLLOW_TABLE.to_csv(var_csv, index=False)
NETWORK_NODE_TABLE.to_csv(node_csv, index=False)
NETWORK_EDGE_TABLE.to_csv(edge_csv, index=False)

print("Saved outputs:")
print(f"- Pair summary: {pair_csv}")
print(f"- Variable summary: {var_csv}")
print(f"- Node table: {node_csv}")
print(f"- Edge table: {edge_csv}")


## Refinement: meaningful uncertainty axis

This refinement recomputes uncertainty from setup-level lag variability (windows, max-lag, direction agreement, and bootstrap spread), then rebuilds the node y-axis and edge thickness scaling for stronger interpretability.

In [ ]:
# Refine uncertainty so y-axis is informative (without rerunning all pair correlations)
from matplotlib.patches import FancyArrowPatch

if "PAIR_SETUP_RESULTS" not in globals() or "PAIR_LEADFOLLOW_TABLE" not in globals():
    raise ValueError("Run the all-pairs cell first so PAIR_SETUP_RESULTS and PAIR_LEADFOLLOW_TABLE exist.")

pair_setup_df = PAIR_SETUP_RESULTS.copy()
pair_summary_df = PAIR_LEADFOLLOW_TABLE.copy()

def _iqr(v: pd.Series) -> float:
    a = v.dropna().to_numpy(dtype=float)
    if len(a) == 0:
        return np.nan
    return float(np.percentile(a, 75) - np.percentile(a, 25))

def _robust_01(x: pd.Series, qlo: float = 0.05, qhi: float = 0.95) -> pd.Series:
    arr = x.to_numpy(dtype=float)
    if len(arr) == 0:
        return pd.Series(dtype=float, index=x.index)
    lo = float(np.nanquantile(arr, qlo))
    hi = float(np.nanquantile(arr, qhi))
    if np.isclose(hi, lo):
        rk = x.rank(method="average", pct=True).fillna(0.5)
        return rk
    clipped = np.clip(arr, lo, hi)
    return pd.Series((clipped - lo) / (hi - lo), index=x.index)

comp_rows = []
for _, row in pair_summary_df.iterrows():
    x_col = row["var_x"]
    y_col = row["var_y"]
    sdf = pair_setup_df[(pair_setup_df["var_x"] == x_col) & (pair_setup_df["var_y"] == y_col)].copy()
    if sdf.empty:
        continue

    lag_equiv = sdf["lag_x_to_y_equiv"]
    setup_iqr = _iqr(lag_equiv)

    by_window = sdf.groupby("window_halfwidth_days")["lag_x_to_y_equiv"].median()
    by_maxlag = sdf.groupby("max_lag_days")["lag_x_to_y_equiv"].median()
    window_iqr = _iqr(by_window)
    maxlag_iqr = _iqr(by_maxlag)

    pvt = sdf.pivot_table(
        index=["window_halfwidth_days", "max_lag_days"],
        columns="direction",
        values="lag_x_to_y_equiv",
        aggfunc="median",
    )
    if {"x->y", "y->x"}.issubset(set(pvt.columns)):
        dir_disagreement = float(np.median(np.abs(pvt["x->y"] - pvt["y->x"])))
    else:
        dir_disagreement = np.nan

    sign_instability = 1.0 - float(row.get("sign_consistency", np.nan))
    boot_iqr = float(row.get("bootstrap_iqr_days", np.nan))
    boot_ci_width = float(row.get("bootstrap_ci_width", np.nan))

    comp_rows.append({
        "var_x": x_col,
        "var_y": y_col,
        "u_setup_iqr": setup_iqr,
        "u_window_iqr": window_iqr,
        "u_maxlag_iqr": maxlag_iqr,
        "u_dir_disagreement": dir_disagreement,
        "u_sign_instability": sign_instability,
        "u_boot_iqr": boot_iqr,
        "u_boot_ci_width": boot_ci_width,
    })

comp_df = pd.DataFrame(comp_rows)
if comp_df.empty:
    raise ValueError("Could not compute uncertainty components.")

for col in [
    "u_setup_iqr", "u_window_iqr", "u_maxlag_iqr",
    "u_dir_disagreement", "u_sign_instability", "u_boot_iqr", "u_boot_ci_width",
]:
    comp_df[f"{col}_n"] = _robust_01(comp_df[col].fillna(comp_df[col].median()))

comp_df["uncertainty_composite"] = (
    0.20 * comp_df["u_setup_iqr_n"] +
    0.18 * comp_df["u_window_iqr_n"] +
    0.18 * comp_df["u_maxlag_iqr_n"] +
    0.12 * comp_df["u_dir_disagreement_n"] +
    0.10 * comp_df["u_sign_instability_n"] +
    0.12 * comp_df["u_boot_iqr_n"] +
    0.10 * comp_df["u_boot_ci_width_n"]
)

pair_summary_df = pair_summary_df.merge(
    comp_df[["var_x", "var_y", "uncertainty_composite"]],
    on=["var_x", "var_y"],
    how="left",
)

pair_summary_df["uncertainty_norm"] = _robust_01(pair_summary_df["uncertainty_composite"]).astype(float)
pair_summary_df["certainty_norm"] = 1.0 - pair_summary_df["uncertainty_norm"]
pair_summary_df["edge_width"] = 0.6 + 5.0 * np.power(pair_summary_df["certainty_norm"], 1.15)
pair_summary_df["edge_alpha"] = 0.15 + 0.75 * np.power(pair_summary_df["certainty_norm"], 0.9)

vars_all = sorted(set(pair_summary_df["var_x"]).union(set(pair_summary_df["var_y"])))
lead_counts = pair_summary_df["leader"].value_counts()
follow_counts = pair_summary_df["follower"].value_counts()

var_rows = []
for v in vars_all:
    n_lead = int(lead_counts.get(v, 0))
    n_follow = int(follow_counts.get(v, 0))
    n_total = n_lead + n_follow
    lead_fraction = float(n_lead / n_total) if n_total > 0 else np.nan
    follow_fraction = float(n_follow / n_total) if n_total > 0 else np.nan
    lead_tendency = float((n_lead - n_follow) / n_total) if n_total > 0 else np.nan

    involved = pair_summary_df[(pair_summary_df["leader"] == v) | (pair_summary_df["follower"] == v)]
    if involved.empty:
        var_unc_raw = np.nan
    else:
        med_u = float(involved["uncertainty_norm"].median())
        iqr_u = _iqr(involved["uncertainty_norm"])
        var_unc_raw = med_u + 0.6 * (0.0 if np.isnan(iqr_u) else iqr_u)

    var_rows.append({
        "variable": v,
        "lead_count": n_lead,
        "follow_count": n_follow,
        "n_pairs_involved": n_total,
        "lead_fraction": lead_fraction,
        "follow_fraction": follow_fraction,
        "lead_tendency": lead_tendency,
        "uncertainty_raw": var_unc_raw,
    })

var_summary_df = pd.DataFrame(var_rows)
var_summary_df["uncertainty_norm_base"] = _robust_01(var_summary_df["uncertainty_raw"].fillna(var_summary_df["uncertainty_raw"].median()))
var_summary_df["uncertainty_rank"] = var_summary_df["uncertainty_raw"].rank(method="average", pct=True).fillna(0.5)
var_summary_df["uncertainty_norm"] = 0.65 * var_summary_df["uncertainty_norm_base"] + 0.35 * var_summary_df["uncertainty_rank"]
var_summary_df = var_summary_df.sort_values(["lead_tendency", "uncertainty_norm"], ascending=[False, True])

node_df = var_summary_df.copy()
node_df["x"] = -node_df["lead_tendency"]
node_df["y"] = node_df["uncertainty_norm"]

edge_df = pair_summary_df.copy()
edge_df["source"] = edge_df["leader"]
edge_df["target"] = edge_df["follower"]
edge_df = edge_df.sort_values(["certainty_norm", "median_abs_r"], ascending=[False, False]).head(int(ALLPAIR_EDGE_TOP_N))

plot_nodes = node_df.set_index("variable")
fig, ax = plt.subplots(figsize=(13, 7))
for _, e in edge_df.iterrows():
    s = e["source"]
    t = e["target"]
    if s not in plot_nodes.index or t not in plot_nodes.index:
        continue
    x1, y1 = float(plot_nodes.loc[s, "x"]), float(plot_nodes.loc[s, "y"])
    x2, y2 = float(plot_nodes.loc[t, "x"]), float(plot_nodes.loc[t, "y"])
    rad = 0.10 if (y2 - y1) >= 0 else -0.10
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle="-|>",
        mutation_scale=9 + 6 * float(e["certainty_norm"]),
        linewidth=float(e["edge_width"]),
        color="#2a9d8f" if float(e["median_lag_x_to_y"]) * float(e["median_abs_r"]) >= 0 else "#e76f51",
        alpha=float(e["edge_alpha"]),
        shrinkA=12,
        shrinkB=12,
    )
    ax.add_patch(arrow)

sc = ax.scatter(
    node_df["x"], node_df["y"],
    s=180 + 260 * node_df["n_pairs_involved"] / max(1, node_df["n_pairs_involved"].max()),
    c=node_df["lead_tendency"],
    cmap="coolwarm_r",
    edgecolors="white",
    linewidths=1.2,
    zorder=3,
)

for _, r in node_df.iterrows():
    ax.text(float(r["x"]), float(r["y"]) + 0.02, str(r["variable"]).replace("_sg", ""), ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Leading tendency (left = more leading, right = more following)")
ax.set_ylabel("Uncertainty (top = higher)")
ax.set_title("All-variable lead/follow map with composite uncertainty")
ax.grid(alpha=0.25)
ax.axvline(0, color="#777777", linestyle="--", linewidth=1.0)
ax.set_ylim(-0.02, 1.05)
cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Lead tendency score")
plt.tight_layout()
plt.show()

PAIR_LEADFOLLOW_TABLE = pair_summary_df.copy()
VARIABLE_LEADFOLLOW_TABLE = var_summary_df.copy()
NETWORK_NODE_TABLE = node_df.copy()
NETWORK_EDGE_TABLE = edge_df.copy()

pair_csv = DATA_DIR / "processed" / "allpairs_lead_follow_uncertainty_2025.csv"
var_csv = DATA_DIR / "processed" / "variable_lead_follow_counts_2025.csv"
node_csv = DATA_DIR / "processed" / "network_nodes_lead_uncertainty_2025.csv"
edge_csv = DATA_DIR / "processed" / "network_edges_lead_uncertainty_2025.csv"
PAIR_LEADFOLLOW_TABLE.to_csv(pair_csv, index=False)
VARIABLE_LEADFOLLOW_TABLE.to_csv(var_csv, index=False)
NETWORK_NODE_TABLE.to_csv(node_csv, index=False)
NETWORK_EDGE_TABLE.to_csv(edge_csv, index=False)

print("Refined uncertainty saved.")
print(f"- {pair_csv}")
print(f"- {var_csv}")
print(f"- {node_csv}")
print(f"- {edge_csv}")

print("\nUncertainty component ranges (pair-level):")
display(comp_df[[
    "u_setup_iqr", "u_window_iqr", "u_maxlag_iqr",
    "u_dir_disagreement", "u_sign_instability", "u_boot_iqr", "u_boot_ci_width",
]].describe().T)

print("\nUpdated variable summary:")
display(VARIABLE_LEADFOLLOW_TABLE[[
    "variable", "lead_count", "follow_count", "lead_tendency", "uncertainty_norm",
]].sort_values(["uncertainty_norm", "lead_tendency"], ascending=[False, False]))

## Core-anchored lead/follow (satellite-stable core structure)

This section keeps the non-satellite structure fixed (core-only pairwise results), while still using the full all-pairs table for satellite-aware context.

What it adds:
- Core-only lead tendency for non-satellite variables.
- Augmented (all-variable) lead tendency for comparison.
- Anchored lead tendency: core variables use core-only tendency; satellite variables keep augmented tendency.
- Quick VPD check and comparison plots/tables.

In [ ]:
# Core-anchored lead/follow mapping: stabilize non-satellite structure
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

if "PAIR_LEADFOLLOW_TABLE" not in globals():
    raise ValueError("Run the all-pairs cell first so PAIR_LEADFOLLOW_TABLE exists.")

pair_all = PAIR_LEADFOLLOW_TABLE.copy()
if pair_all.empty:
    raise ValueError("PAIR_LEADFOLLOW_TABLE is empty.")

# Resolve variable families
if "ALLPAIR_PREP_METADATA" in globals() and not ALLPAIR_PREP_METADATA.empty:
    _meta = ALLPAIR_PREP_METADATA.copy()
    sat_vars = set(_meta.loc[_meta["family"] == "satellite", "column"].astype(str))
    all_vars = set(_meta["column"].astype(str))
else:
    all_vars = set(pair_all["var_x"].astype(str)).union(set(pair_all["var_y"].astype(str)))
    sat_vars = {v for v in all_vars if str(v).lower().startswith(("s1_", "s2_"))}

core_vars = sorted(all_vars - sat_vars)
sat_vars = sorted(sat_vars)

if len(core_vars) < 2:
    raise ValueError("Not enough core variables to compute a core-only anchor.")

def _summarize_variables(pair_df: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
    lead_counts = pair_df["leader"].value_counts() if not pair_df.empty else pd.Series(dtype=float)
    follow_counts = pair_df["follower"].value_counts() if not pair_df.empty else pd.Series(dtype=float)

    rows = []
    for v in variables:
        n_lead = int(lead_counts.get(v, 0))
        n_follow = int(follow_counts.get(v, 0))
        n_total = n_lead + n_follow
        lead_tendency = float((n_lead - n_follow) / n_total) if n_total > 0 else np.nan

        involved = pair_df[(pair_df["leader"] == v) | (pair_df["follower"] == v)]
        var_unc = float(involved["uncertainty_norm"].median()) if (not involved.empty and "uncertainty_norm" in involved.columns) else np.nan

        rows.append({
            "variable": v,
            "lead_count": n_lead,
            "follow_count": n_follow,
            "n_pairs_involved": n_total,
            "lead_tendency": lead_tendency,
            "uncertainty_norm": var_unc,
        })
    return pd.DataFrame(rows)

core_pair_mask = pair_all["var_x"].isin(core_vars) & pair_all["var_y"].isin(core_vars)
pair_core = pair_all.loc[core_pair_mask].copy()

core_summary = _summarize_variables(pair_core, core_vars).rename(columns={
    "lead_tendency": "lead_tendency_core",
    "uncertainty_norm": "uncertainty_core",
    "n_pairs_involved": "n_pairs_core",
})

aug_summary = _summarize_variables(pair_all, sorted(all_vars)).rename(columns={
    "lead_tendency": "lead_tendency_aug",
    "uncertainty_norm": "uncertainty_aug",
    "n_pairs_involved": "n_pairs_aug",
})

anchor = aug_summary.merge(
    core_summary[["variable", "lead_tendency_core", "uncertainty_core", "n_pairs_core"]],
    on="variable",
    how="left",
)
anchor["is_satellite"] = anchor["variable"].isin(sat_vars)
anchor["lead_tendency_anchor"] = np.where(
    anchor["is_satellite"],
    anchor["lead_tendency_aug"],
    anchor["lead_tendency_core"],
)
anchor["uncertainty_anchor"] = np.where(
    anchor["is_satellite"],
    anchor["uncertainty_aug"],
    anchor["uncertainty_core"],
)
anchor["x_anchor"] = -anchor["lead_tendency_anchor"]
anchor["y_anchor"] = anchor["uncertainty_anchor"]

# ── Manual position fine-tuning ───────────────────────────────────────────
# X nudge: positive = shift right (later); negative = shift left (earlier).
# Y nudge: positive = shift up; negative = shift down.
# Values are ADDED to the computed anchor; 0.0 = no change.
# x_anchor raw values (from cross-correlation lead tendency, negated):
#   tair_c_mean_sg          = -0.81  |  vpd_hpa_max_sg           = -0.81
#   leaf_angle_cam67_mean_sg= -0.71  |  leaf_angle_cam65_mean_sg = -0.71
#   twd_um_mean_sg          = -0.62  |  swp_mpa_predawn_mean_sg  = -0.43
#   vod_mean_sg             = -0.33  |  et_mmd_mean_sg           = -0.33
#   s2_kndvi_savgol         = -0.14  |  s2_evi_savgol            = -0.05
#   s2_nirv_savgol          = +0.05  |  s1_cr_savgol             = +0.14
#   s2_savi_savgol          = +0.14  |  sapflow_jscm3cm2d_mean_sg= +0.43
#   gpp_umolm2s_mean_sg     = +0.43  |  s2_ndii_savgol           = +0.43
#   s2_ndvi_savgol          = +0.62  |  gcc_p90_mean_sg          = +0.71
#   s2_mtci_savgol          = +0.81  |  pai_total_sg             = +0.81
#   s2_cire_savgol          = +0.90  |  s1_span_savgol           = -0.52

BREAKPOINT_X_NUDGE = {
    # ── Atmosphere ──────────────────────────────────────────
    "tair_c_mean_sg":               +0.0,   # raw x = -0.81
    "vpd_hpa_max_sg":               +0.0,   # raw x = -0.81
    # ── Plant hydraulics ────────────────────────────────────
    "twd_um_mean_sg":               +0.0,   # raw x = -0.62
    "swp_mpa_predawn_mean_sg":      +0.0,   # raw x = -0.43
    "sapflow_jscm3cm2d_mean_sg":    -0.2,   # raw x = +0.43
    # ── Canopy dynamics ─────────────────────────────────────
    "leaf_angle_cam67_mean_sg":     +0.05,   # raw x = -0.71
    "leaf_angle_cam65_mean_sg":     +0.25,   # raw x = -0.71
    "vod_mean_sg":                  +0.0,   # raw x = -0.33
    "pai_total_sg":                 +0.3,   # raw x = +0.81
    "gcc_p90_mean_sg":              +0.0,   # raw x = +0.71
    # ── Ecosystem fluxes ────────────────────────────────────
    "et_mmd_mean_sg":               +0.0,   # raw x = -0.33
    "gpp_umolm2s_mean_sg":          +0.0,   # raw x = +0.43
    # ── Satellite — Sentinel-1 ──────────────────────────────
    "s1_cr_savgol":                 +0.35,   # raw x = +0.14
    "s1_span_savgol":               +0.2,   # raw x = -0.52
    # ── Satellite — Sentinel-2 ──────────────────────────────
    "s2_cire_savgol":               +0.0,   # raw x = +0.90
    "s2_evi_savgol":                +0.15,   # raw x = -0.05
    "s2_kndvi_savgol":              -0.25,   # raw x = -0.14
    "s2_mtci_savgol":               +0.0,   # raw x = +0.81
    "s2_ndii_savgol":               +0.0,   # raw x = +0.43
    "s2_ndvi_savgol":               +0.0,   # raw x = +0.62
    "s2_nirv_savgol":               +0.0,   # raw x = +0.05
    "s2_savi_savgol":               +0.0,   # raw x = +0.14
}

BREAKPOINT_Y_NUDGE = {
    # ── Atmosphere ──────────────────────────────────────────
    "tair_c_mean_sg":               +0.0,   # raw y = 0.98
    "vpd_hpa_max_sg":               +0.0,   # raw y = 1.00
    # ── Plant hydraulics ────────────────────────────────────
    "twd_um_mean_sg":               +0.0,   # raw y = 0.43
    "swp_mpa_predawn_mean_sg":      +0.0,   # raw y = 0.45
    "sapflow_jscm3cm2d_mean_sg":    +0.0,   # raw y = 0.14
    # ── Canopy dynamics ─────────────────────────────────────
    "leaf_angle_cam67_mean_sg":     +0.0,   # raw y = 0.23
    "leaf_angle_cam65_mean_sg":     +0.0,   # raw y = 0.39
    "vod_mean_sg":                  +0.0,   # raw y = 0.27
    "pai_total_sg":                 +0.0,   # raw y = 0.84
    "gcc_p90_mean_sg":              +0.0,   # raw y = 0.03
    # ── Ecosystem fluxes ────────────────────────────────────
    "et_mmd_mean_sg":               +0.0,   # raw y = 0.70
    "gpp_umolm2s_mean_sg":          +0.0,   # raw y = 0.15
    # ── Satellite — Sentinel-1 ──────────────────────────────
    "s1_cr_savgol":                 +0.0,   # raw y = 0.07
    "s1_span_savgol":               +0.0,   # raw y = 0.88
    # ── Satellite — Sentinel-2 ──────────────────────────────
    "s2_cire_savgol":               +0.0,   # raw y = 0.22
    "s2_evi_savgol":                +0.0,   # raw y = 0.49
    "s2_kndvi_savgol":              +0.0,   # raw y = 0.62
    "s2_mtci_savgol":               +0.0,   # raw y = 0.30
    "s2_ndii_savgol":               +0.0,   # raw y = 0.36
    "s2_ndvi_savgol":               +0.0,   # raw y = 0.02
    "s2_nirv_savgol":               +0.0,   # raw y = 0.71
    "s2_savi_savgol":               +0.0,   # raw y = 0.67
}

for _var, _nudge in BREAKPOINT_X_NUDGE.items():
    _mask = anchor["variable"] == _var
    if _mask.any() and _nudge != 0.0:
        anchor.loc[_mask, "x_anchor"] = (anchor.loc[_mask, "x_anchor"] + _nudge).clip(-1.0, 1.0)

for _var, _nudge in BREAKPOINT_Y_NUDGE.items():
    _mask = anchor["variable"] == _var
    if _mask.any() and _nudge != 0.0:
        anchor.loc[_mask, "y_anchor"] = anchor.loc[_mask, "y_anchor"] + _nudge

_nonzero_x = {k: v for k, v in BREAKPOINT_X_NUDGE.items() if v != 0.0}
_nonzero_y = {k: v for k, v in BREAKPOINT_Y_NUDGE.items() if v != 0.0}
if _nonzero_x or _nonzero_y:
    print("Position nudges applied:")
    for _var in set(list(_nonzero_x.keys()) + list(_nonzero_y.keys())):
        _row = anchor[anchor["variable"] == _var]
        if not _row.empty:
            _nx = _nonzero_x.get(_var, 0.0)
            _ny = _nonzero_y.get(_var, 0.0)
            print(f"  {_var:40s}  x_nudge={_nx:+.2f}  y_nudge={_ny:+.2f}  → x={float(_row['x_anchor'].iloc[0]):.3f}  y={float(_row['y_anchor'].iloc[0]):.3f}")
else:
    print("No position nudges applied (all zero).")

# Fallback for missing core uncertainty values
if anchor["y_anchor"].isna().any():
    anchor["y_anchor"] = anchor["y_anchor"].fillna(anchor["uncertainty_aug"]).fillna(anchor["y_anchor"].median())

# Diagnostic table
compare = anchor[[
    "variable", "is_satellite", "lead_tendency_core", "lead_tendency_aug", "lead_tendency_anchor",
    "n_pairs_core", "n_pairs_aug",
]].copy()
compare["delta_aug_minus_core"] = compare["lead_tendency_aug"] - compare["lead_tendency_core"]

print(f"\nCore variables: {len(core_vars)} | Satellite variables: {len(sat_vars)}")
print(f"Core-core pairs: {len(pair_core)} | All pairs: {len(pair_all)}")

vpd_candidates = [v for v in compare["variable"] if "vpd" in str(v).lower()]
if vpd_candidates:
    print("\nVPD anchor check:")
    display(compare[compare["variable"].isin(vpd_candidates)].sort_values("variable"))

print("\nLargest core shifts after adding satellite variables:")
display(
    compare[~compare["is_satellite"]]
    .sort_values("delta_aug_minus_core", key=lambda s: s.abs(), ascending=False)
    .head(15)
    [["variable", "lead_tendency_core", "lead_tendency_aug", "lead_tendency_anchor", "delta_aug_minus_core"]]
 )

# ── Compartment colour map ─────────────────────────────────────────────────
COMPARTMENT_COLOR = {
    "atmosphere":        "#a65984",
    #"soil":              "#6b59a6",
    "plant hydraulics":  "#59a3a6",
    "canopy dynamics":   "#66a659",
    "ecosystem fluxes":  "#a68a59",
    "satellite":         "#7F7F7F",
}

VAR_COMPARTMENT = {
    "vpd_hpa_max_sg":               "atmosphere",
    "tair_c_mean_sg":               "atmosphere",
    "sapflow_jscm3cm2d_mean_sg":    "plant hydraulics",
    "swp_mpa_predawn_mean_sg":      "plant hydraulics",
    "twd_um_mean_sg":               "plant hydraulics",
    "leaf_angle_cam67_mean_sg":     "canopy dynamics",
    "leaf_angle_cam65_mean_sg":     "canopy dynamics",
    "pai_total_sg":                 "canopy dynamics",
    "vod_mean_sg":                  "canopy dynamics",
    "gcc_p90_mean_sg":              "canopy dynamics",
    "et_mmd_mean_sg":               "ecosystem fluxes",
    "gpp_umolm2s_mean_sg":          "ecosystem fluxes",
    "s1_cr_savgol":                 "satellite",
    "s1_span_savgol":               "satellite",
    "s2_cire_savgol":               "satellite",
    "s2_evi_savgol":                "satellite",
    "s2_kndvi_savgol":              "satellite",
    "s2_mtci_savgol":               "satellite",
    "s2_ndii_savgol":               "satellite",
    "s2_ndvi_savgol":               "satellite",
    "s2_nirv_savgol":               "satellite",
    "s2_savi_savgol":               "satellite",
}

def _node_color_for(v: str) -> str:
    compartment = VAR_COMPARTMENT.get(str(v))
    if compartment is not None:
        return COMPARTMENT_COLOR[compartment]
    # Fallback: satellite prefix → satellite colour, otherwise grey
    v_low = str(v).lower()
    if v_low.startswith(("s1_", "s2_")):
        return COMPARTMENT_COLOR["satellite"]
    return "#aaaaaa"

# User-adjustable design settings
STYLE = {
    "min_neighbors_per_node": 10,
    "extra_top_edges": 70,
    "distance_y_weight": 0.25,
    "edge_curve_rad": 0.2,
    "edge_width_min": 0.5,
    "edge_width_max": 1.5,
    "edge_alpha_min": 0.3,
    "edge_alpha_max": 0.6,
    "node_size_min": 400,
    "node_size_span": 480,
    "label_fontsize": 13,
    "label_box_alpha": 0.8,
    "label_box_pad": 0.15,
    "y_jitter": 0.012,
    "min_node_gap": 0.010,
    "repel_iters": 320,
    "repel_y_pad": 0.5,
    "hide_y_axis": True,
    "figsize_network": (12.0, 8.0),
    "figsize_drift": (8.2, 6.8),
}

PALETTE = {
    "edge_positive": "#1f77b4",
    "edge_negative": "#d55e00",
    "reference": "#808080",
    "background": "#ffffff",
    "text": "#1f2937",
}

DISPLAY_NAME_MAP = {
    "tair_c_mean_sg":  r"$\mathrm{T_{air}}$ ",
    "vpd_hpa_max_sg": r"$\mathrm{VPD}_{\max}$",
    "swp_mpa_predawn_mean_sg": r"$\Psi_\mathrm{Stem}$",
    "sapflow_jscm3cm2d_mean_sg": r"$\mathrm{J_s}$",
    "twd_um_mean_sg": r"$\mathrm{TWD}$",
    "vod_mean_sg": r"$\mathrm{VOD}$",
    "leaf_angle_cam65_mean_sg": r"$\mathrm{Leaf\ angle\ (3.5m)}$",
    "leaf_angle_cam67_mean_sg": r"$\mathrm{Leaf\ angle\ (8.5m)}$",
    "pai_total_sg": r"$\mathrm{PAI}$",
    "gcc_p90_mean_sg": r"$\mathrm{GCC}$",
    "gpp_umolm2s_mean_sg": r"$\mathrm{GPP}$",
    "et_mmd_mean_sg": r"$\mathrm{ET}$",
}

def _pretty_name(v: str) -> str:
    v = str(v)
    if v in DISPLAY_NAME_MAP:
        return DISPLAY_NAME_MAP[v]
    if v.startswith("s1_"):
        metric = v.replace("s1_", "").replace("_savgol", "").replace("_raw", "").upper()
        return f"S1 {metric}"
    if v.startswith("s2_"):
        metric = v.replace("s2_", "").replace("_savgol", "").replace("_raw", "").upper()
        return f"S2 {metric}"
    return (
        v.replace("_sg", "")
        .replace("_mean", "")
        .replace("_std", "")
        .replace("_", " ")
        .strip()
        .title()
    )

def _enforce_min_distance_y(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    radius_col: str,
    min_gap: float,
    iters: int = 200,
    y_pad: float = 0.30,
 ) -> tuple[pd.DataFrame, float]:
    df = df.copy()
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    r = df[radius_col].to_numpy(dtype=float)
    y_base = df["y_anchor"].to_numpy(dtype=float)
    n = len(df)

    for _ in range(int(iters)):
        moved = False
        for i in range(n):
            for j in range(i + 1, n):
                dx = x[j] - x[i]
                dy = y[j] - y[i]
                dist = float(np.hypot(dx, dy))
                target = float(r[i] + r[j] + min_gap)
                if dist < target:
                    overlap = target - dist
                    if abs(dy) < 1e-9:
                        sign = 1.0 if ((i + j) % 2 == 0) else -1.0
                    else:
                        sign = 1.0 if dy > 0 else -1.0
                    shift = 0.5 * overlap * sign
                    y[j] += shift
                    y[i] -= shift
                    moved = True

        # Allow a wide hidden y-band for spacing, since y-axis is not shown.
        ymin = float(np.nanmin(y_base) - y_pad)
        ymax = float(np.nanmax(y_base) + y_pad)
        y = np.clip(y, ymin, ymax)

        if not moved:
            break

    min_margin = np.inf
    for i in range(n):
        for j in range(i + 1, n):
            dij = float(np.hypot(x[j] - x[i], y[j] - y[i]))
            target = float(r[i] + r[j] + min_gap)
            margin = dij - target
            if margin < min_margin:
                min_margin = margin

    df[y_col] = y
    return df, float(min_margin)

anchor["label"] = anchor["variable"].map(_pretty_name)
anchor["node_color"] = anchor["variable"].map(_node_color_for)

# Add deterministic y-jitter for readability in network layout
rng = np.random.default_rng(42)
anchor["y_plot"] = anchor["y_anchor"] + rng.uniform(
    -float(STYLE["y_jitter"]), float(STYLE["y_jitter"]), size=len(anchor)
)

# Build size-aware node radius hint used for minimum-distance layout
anchor["node_size_plot"] = STYLE["node_size_min"] + STYLE["node_size_span"] * anchor["n_pairs_aug"] / max(1, anchor["n_pairs_aug"].max())
size_norm = anchor["node_size_plot"] / max(1.0, float(anchor["node_size_plot"].max()))
anchor["radius_hint"] = 0.028 + 0.070 * size_norm

# Enforce size-aware node spacing after jitter (y-only repulsion; x remains interpretable).
anchor, min_margin_after_repel = _enforce_min_distance_y(
    anchor,
    x_col="x_anchor",
    y_col="y_plot",
    radius_col="radius_hint",
    min_gap=float(STYLE["min_node_gap"]),
    iters=int(STYLE["repel_iters"]),
    y_pad=float(STYLE["repel_y_pad"]),
)
print(f"Node overlap margin after repel: {min_margin_after_repel:.3f} (>=0 means no node overlap target violations)")

# Plot 1: core-only vs augmented for core variables
core_cmp = compare[~compare["is_satellite"]].copy()
core_cmp["node_color"] = core_cmp["variable"].map(_node_color_for)

fig, ax = plt.subplots(figsize=STYLE["figsize_drift"])
ax.set_facecolor(PALETTE["background"])
fig.patch.set_facecolor(PALETTE["background"])

lims = [-1.0, 1.0]
ax.plot(lims, lims, "--", color=PALETTE["reference"], linewidth=1.0, zorder=1)

for _, r in core_cmp.iterrows():
    ax.scatter(
        float(r["lead_tendency_core"]),
        float(r["lead_tendency_aug"]),
        s=90,
        alpha=0.9,
        color=r["node_color"],
        edgecolors="white",
        linewidths=0.8,
        zorder=3,
    )
    lbl = _pretty_name(r["variable"])
    ax.text(
        float(r["lead_tendency_core"]) + 0.016,
        float(r["lead_tendency_aug"]) + 0.016,
        lbl,
        fontsize=9,
        color=PALETTE["text"],
        bbox={
            "boxstyle": f"round,pad=0.10",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.7,
        },
    )

ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Core-only timing tendency", color=PALETTE["text"])
ax.set_ylabel("With-satellite timing tendency", color=PALETTE["text"])
ax.xaxis.grid(True, alpha=0.22)
ax.yaxis.grid(True, alpha=0.22)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(colors=PALETTE["text"])

# Compartment legend (matching network plot)
_drift_handles = [
    mpatches.Patch(facecolor=color, edgecolor="white", label=name.capitalize())
    for name, color in COMPARTMENT_COLOR.items()
]
ax.legend(
    handles=_drift_handles,
    title="",
    loc="upper left",
    fontsize=9,
    framealpha=0.85,
    edgecolor="none",
)

plt.tight_layout()
plt.show()

# Build edge certainty first
pair_all = pair_all.copy()
if "certainty_norm" not in pair_all.columns:
    pair_all["certainty_norm"] = 1.0 - pair_all.get("uncertainty_norm", 0.5)
pair_all["certainty_norm"] = pair_all["certainty_norm"].clip(0.0, 1.0)

def _pair_key(a: str, b: str) -> tuple[str, str]:
    return tuple(sorted((str(a), str(b))))

# Keep best representative edge per unordered pair
pair_lookup = {}
for _, r in pair_all.iterrows():
    a = str(r["var_x"])
    b = str(r["var_y"])
    k = _pair_key(a, b)
    score = (float(r.get("certainty_norm", 0.0)), float(r.get("median_abs_r", 0.0)))
    if (k not in pair_lookup) or (score > pair_lookup[k][0]):
        pair_lookup[k] = (score, r.copy())

nodes_plot = anchor.set_index("variable")
node_names = list(nodes_plot.index)

# 1) nearest-neighbor links: guarantee connectivity per node
selected_keys = set()
for v in node_names:
    xv = float(nodes_plot.loc[v, "x_anchor"])

    yv = float(nodes_plot.loc[v, "y_plot"])

    drows = []
    for u in node_names:
        if u == v:
            continue
        xu = float(nodes_plot.loc[u, "x_anchor"])

        yu = float(nodes_plot.loc[u, "y_plot"])

        d = np.sqrt((xu - xv) ** 2 + STYLE["distance_y_weight"] * (yu - yv) ** 2)
        drows.append((u, d))
    drows = sorted(drows, key=lambda t: t[1])[: int(STYLE["min_neighbors_per_node"])]
    for u, _ in drows:
        selected_keys.add(_pair_key(v, u))

# 2) add globally strongest edges for richer connected structure
sorted_all = pair_all.sort_values(["certainty_norm", "median_abs_r"], ascending=[False, False])
extra_added = 0
for _, r in sorted_all.iterrows():
    k = _pair_key(r["var_x"], r["var_y"])

    if k in pair_lookup and k not in selected_keys:
        selected_keys.add(k)
        extra_added += 1
        if extra_added >= int(STYLE["extra_top_edges"]):
            break

edge_rows = [pair_lookup[k][1] for k in selected_keys if k in pair_lookup]
edge_plot = pd.DataFrame(edge_rows).copy()
edge_plot["source"] = edge_plot["leader"].astype(str)
edge_plot["target"] = edge_plot["follower"].astype(str)

# Edge width scaled by median R² (stronger fit = thicker line)
_edge_r2 = edge_plot["median_r2"] if "median_r2" in edge_plot.columns else edge_plot["median_abs_r"] ** 2
edge_plot["edge_width"] = (
    STYLE["edge_width_min"] + (STYLE["edge_width_max"] - STYLE["edge_width_min"]) * edge_plot["certainty_norm"]
) * _edge_r2
edge_plot["edge_alpha"] = STYLE["edge_alpha_min"] + (STYLE["edge_alpha_max"] - STYLE["edge_alpha_min"]) * edge_plot["certainty_norm"]

# Degree check (should be >= min neighbors for nearly all variables)
deg = {v: 0 for v in node_names}
for k in selected_keys:
    deg[k[0]] += 1
    deg[k[1]] += 1
print(f"Anchored graph edges: {len(edge_plot)} | min degree: {min(deg.values())} | median degree: {int(np.median(list(deg.values())))}")

# Plot 2: anchored network (x from anchored tendency, y kept for layout but hidden)
fig, ax = plt.subplots(figsize=STYLE["figsize_network"], constrained_layout=True)
ax.set_facecolor(PALETTE["background"])


for _, e in edge_plot.iterrows():
    s = str(e["source"])

    t = str(e["target"])

    if s not in nodes_plot.index or t not in nodes_plot.index:
        continue
    x1, y1 = float(nodes_plot.loc[s, "x_anchor"]), float(nodes_plot.loc[s, "y_plot"])

    x2, y2 = float(nodes_plot.loc[t, "x_anchor"]), float(nodes_plot.loc[t, "y_plot"])

    # Blue = positive median correlation, orange = negative (colorblind-safe).
    r_signed = float(e.get("median_r_signed", e.get("best_r", 0.0)))
    edge_color = PALETTE["edge_positive"] if r_signed >= 0 else PALETTE["edge_negative"]
    rad = STYLE["edge_curve_rad"] if (y2 - y1) >= 0 else -STYLE["edge_curve_rad"]

    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle="-|>",
        mutation_scale=8 + 2.2 * float(e["certainty_norm"]),
        linewidth=float(e["edge_width"]),
        color=edge_color,
        alpha=float(e["edge_alpha"]),
        shrinkA=10,
        shrinkB=10,
    )
    ax.add_patch(arrow)

# Draw nodes using per-compartment colours
node_size = anchor["node_size_plot"]
node_color = anchor["node_color"].tolist()

ax.scatter(
    anchor["x_anchor"],
    anchor["y_plot"],
    s=node_size,
    c=node_color,
    edgecolors="white",
    linewidths=1.0,
    zorder=3,
)

for _, r in anchor.iterrows():
    ax.text(
        float(r["x_anchor"]),
        float(r["y_plot"]) + 0.035,
        str(r["label"]),
        ha="center",
        va="bottom",
        fontsize=STYLE["label_fontsize"],
        color=PALETTE["text"],
        bbox={
            "boxstyle": f"round,pad={STYLE['label_box_pad']}",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": STYLE["label_box_alpha"],
        },
    )

ax.set_xlim(-1.05, 1.05)
ax.axvline(0, color=PALETTE["reference"], linestyle="--", linewidth=1.0)
ax.xaxis.grid(True, alpha=0.22)
ax.yaxis.grid(False)

# Keep no x-axis numbers; keep only conceptual axis label.
ax.set_xticks([])
ax.set_xlabel("Earlier timing tendency <   > Later timing tendency  ")
#ax.set_title("Anchored association network")

if STYLE["hide_y_axis"]:
    ax.set_ylabel("")
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
else:
    ax.set_ylabel("Uncertainty (layout axis)")
    ax.set_ylim(-0.02, 1.05)

# Compartment legend
legend_handles = [
    mpatches.Patch(facecolor=color, edgecolor="white", label=name.capitalize())
    for name, color in COMPARTMENT_COLOR.items()
]
leg1 = ax.legend(
    handles=legend_handles,
    title="",
    loc="upper left",
    fontsize=10,
    title_fontsize=10,
    framealpha=0.85,
    edgecolor="none",
)
ax.add_artist(leg1)

# Correlation sign legend (below compartment legend)
from matplotlib.lines import Line2D
corr_handles = [
    Line2D([0], [0], color=PALETTE["edge_positive"], linewidth=2, label="Positive correlation"),
    Line2D([0], [0], color=PALETTE["edge_negative"], linewidth=2, label="Negative correlation"),
]
ax.legend(
    handles=corr_handles,
    title="",
    loc="lower left",
    fontsize=10,
    title_fontsize=10,
    framealpha=0.85,
    edgecolor="none",
)

# Save network figure — A0 poster ready (300 dpi, vector PDF + raster PNG)
_net_pdf = FIG_DIR / "poster_network_anchored.pdf"
_net_png = FIG_DIR / "poster_network_anchored.png"
fig.savefig(_net_pdf, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
fig.savefig(_net_png, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"Saved: {_net_pdf}")
print(f"Saved: {_net_png}")

plt.show()

print("Edge colors: blue = positive median correlation, orange = negative median correlation")
print("Edge width: scaled by median R² (certainty × goodness of fit)")
print("Node colors: by compartment (atmosphere, soil, plant hydraulics, canopy dynamics, ecosystem fluxes, satellite)")

CORE_PAIR_LEADFOLLOW_TABLE = pair_core.copy()
CORE_VARIABLE_TABLE = core_summary.copy()
ANCHOR_VARIABLE_TABLE = anchor.copy()
ANCHOR_COMPARISON_TABLE = compare.copy()
print("Core-anchored tables created: CORE_PAIR_LEADFOLLOW_TABLE, CORE_VARIABLE_TABLE, ANCHOR_VARIABLE_TABLE, ANCHOR_COMPARISON_TABLE")


# Colors

- atmosphere: "#a65984"
- soil: #6b59a6
- plant hydraulics: #59a3a6
- canopy dynamics: #66a659
- ecosystem fluxes: #a68a59
- satellite: #7F7F7F

- vpd_hpa_max_sg  (atmosphere)
- tair_c_mean_sg  (atmosphere)
- sapflow_jscm3cm2d_mean_sg (plant hydraulics)
- swp_mpa_predawn_mean_sg (plant hydraulics)  
- twd_um_mean_sg (plant hydraulics)
- leaf_angle_cam67_mean_sg  (8.5, canopy dynamics)
- leaf_angle_cam65_mean_sg  (3.5, canopy dynamics)
- pai_total_sg  (canopy dynamics)
- vod_mean_sg  (canopy dynamics)
- gcc_p90_mean_sg  (canopy dynamics)
- et_mmd_mean_sg (ecosystem fluxes)
- gpp_umolm2s_mean_sg (ecosystem fluxes)
- s1_cr_savgol  (satellite)
- s1_span_savgol  (satellite)
- s2_cire_savgol  (satellite)
- s2_evi_savgol  (satellite)
- s2_kndvi_savgol  (satellite)
- s2_mtci_savgol  (satellite)
- s2_ndii_savgol  (satellite)
- s2_ndvi_savgol  (satellite)
- s2_nirv_savgol  (satellite)
- s2_savi_savgol  (satellite)

In [ ]:
# Diagnostic checks for anchored logic correctness
import numpy as np
import pandas as pd

required = [
    "PAIR_LEADFOLLOW_TABLE",
    "CORE_PAIR_LEADFOLLOW_TABLE",
    "CORE_VARIABLE_TABLE",
    "ANCHOR_VARIABLE_TABLE",
    "ANCHOR_COMPARISON_TABLE",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Run Cell 33 first; missing: {missing}")

pair_all_diag = PAIR_LEADFOLLOW_TABLE.copy()
pair_core_diag = CORE_PAIR_LEADFOLLOW_TABLE.copy()
core_tbl = CORE_VARIABLE_TABLE.copy()
anchor_tbl = ANCHOR_VARIABLE_TABLE.copy()
cmp_tbl = ANCHOR_COMPARISON_TABLE.copy()

def summarize(df: pd.DataFrame, variables: list[str], out_col: str) -> pd.DataFrame:
    lc = df["leader"].value_counts()
    fc = df["follower"].value_counts()
    out = []
    for v in variables:
        n_lead = int(lc.get(v, 0))
        n_follow = int(fc.get(v, 0))
        n = n_lead + n_follow
        val = (n_lead - n_follow) / n if n > 0 else np.nan
        out.append({"variable": v, out_col: val})
    return pd.DataFrame(out)

core_vars_diag = sorted(cmp_tbl.loc[~cmp_tbl["is_satellite"], "variable"].astype(str).tolist())
sat_vars_diag = sorted(cmp_tbl.loc[cmp_tbl["is_satellite"], "variable"].astype(str).tolist())

# 1) Core-core correctness: anchored core values should equal core-only summary exactly
re_core = summarize(pair_core_diag, core_vars_diag, "re_core")
m1 = core_tbl[["variable", "lead_tendency_core"]].merge(re_core, on="variable", how="inner")
core_recompute_max_abs_err = float((m1["lead_tendency_core"] - m1["re_core"]).abs().max()) if not m1.empty else np.nan

# 2) Stitching rule correctness in the final anchor table
check = anchor_tbl[["variable", "is_satellite", "lead_tendency_core", "lead_tendency_aug", "lead_tendency_anchor"]].copy()
core_rule_ok = bool(np.allclose(
    check.loc[~check["is_satellite"], "lead_tendency_anchor"],
    check.loc[~check["is_satellite"], "lead_tendency_core"],
    equal_nan=True,
))
sat_rule_ok = bool(np.allclose(
    check.loc[check["is_satellite"], "lead_tendency_anchor"],
    check.loc[check["is_satellite"], "lead_tendency_aug"],
    equal_nan=True,
))

# 3) Satellite-satellite internal-only comparison (not used by current implementation)
pair_sat_sat = pair_all_diag[pair_all_diag["var_x"].isin(sat_vars_diag) & pair_all_diag["var_y"].isin(sat_vars_diag)].copy()
sat_aug = summarize(pair_all_diag, sat_vars_diag, "sat_aug")
sat_internal = summarize(pair_sat_sat, sat_vars_diag, "sat_internal")
sat_cmp = sat_aug.merge(sat_internal, on="variable", how="left")
sat_cmp["abs_diff_aug_vs_satonly"] = (sat_cmp["sat_aug"] - sat_cmp["sat_internal"]).abs()

sat_only_pairs = len(pair_sat_sat)
sat_internal_available = int(sat_cmp["sat_internal"].notna().sum())
sat_median_abs_diff = float(sat_cmp["abs_diff_aug_vs_satonly"].median(skipna=True)) if sat_internal_available > 0 else np.nan
sat_max_abs_diff = float(sat_cmp["abs_diff_aug_vs_satonly"].max(skipna=True)) if sat_internal_available > 0 else np.nan

print("=== Anchor Logic Diagnostics ===")
print(f"Core recompute max abs error: {core_recompute_max_abs_err:.6g}")
print(f"Stitching rule core->core-only correct: {core_rule_ok}")
print(f"Stitching rule sat->augmented correct: {sat_rule_ok}")
print(f"Satellite-satellite pair count: {sat_only_pairs}")
print(f"Sat variables with sat-only estimate: {sat_internal_available}/{len(sat_vars_diag)}")
print(f"Median |aug - sat_only| for satellites: {sat_median_abs_diff:.3f}")
print(f"Max |aug - sat_only| for satellites: {sat_max_abs_diff:.3f}")

print("\nTop satellite differences (augmented minus sat-only):")
display(
    sat_cmp.sort_values("abs_diff_aug_vs_satonly", ascending=False)
    [["variable", "sat_aug", "sat_internal", "abs_diff_aug_vs_satonly"]]
    .head(10)
)

In [ ]:
# Network stitching diagnostic: requested neighbor keys vs actual available pair edges
if not {"selected_keys", "pair_lookup", "edge_plot", "anchor"}.issubset(set(globals().keys())):
    raise ValueError("Run Cell 33 first to populate selected_keys, pair_lookup, edge_plot, and anchor.")

requested = len(selected_keys)
available = sum(1 for k in selected_keys if k in pair_lookup)
missing = requested - available

node_names_diag = anchor["variable"].astype(str).tolist()
deg_actual = {v: 0 for v in node_names_diag}
for _, row in edge_plot.iterrows():
    s = str(row["source"]); t = str(row["target"])

    if s in deg_actual:
        deg_actual[s] += 1
    if t in deg_actual:
        deg_actual[t] += 1

print("=== Network Stitching Diagnostics ===")
print(f"Requested neighbor keys: {requested}")
print(f"Available in pair table: {available}")
print(f"Missing requested keys: {missing}")
print(f"Actual edge_plot min degree: {min(deg_actual.values())}")
print(f"Actual edge_plot median degree: {int(np.median(list(deg_actual.values())))}")

In [ ]:
# Two-track comparison: mixed-satellite anchor vs satellite-only anchor
from matplotlib.patches import FancyArrowPatch

required_vars = {
    "PAIR_LEADFOLLOW_TABLE",
    "ANCHOR_VARIABLE_TABLE",
    "ANCHOR_COMPARISON_TABLE",
    "STYLE",
    "PALETTE",
}
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Run Cell 33 first; missing globals: {missing_vars}")

pair_all_cmp = PAIR_LEADFOLLOW_TABLE.copy()
anchor_base = ANCHOR_VARIABLE_TABLE.copy()
cmp_tbl = ANCHOR_COMPARISON_TABLE.copy()

if "_pretty_name" not in globals():
    def _pretty_name(v: str) -> str:
        return str(v)

if "_enforce_min_distance_y" not in globals():
    def _enforce_min_distance_y(df, x_col, y_col, radius_col, min_gap, iters=200, y_pad=0.30):
        df = df.copy()
        x = df[x_col].to_numpy(dtype=float)
        y = df[y_col].to_numpy(dtype=float)
        r = df[radius_col].to_numpy(dtype=float)
        y_base = df["y_anchor"].to_numpy(dtype=float)
        n = len(df)
        for _ in range(int(iters)):
            moved = False
            for i in range(n):
                for j in range(i + 1, n):
                    dx = x[j] - x[i]
                    dy = y[j] - y[i]
                    dist = float(np.hypot(dx, dy))
                    target = float(r[i] + r[j] + min_gap)
                    if dist < target:
                        overlap = target - dist
                        sign = 1.0 if (abs(dy) < 1e-9 and ((i + j) % 2 == 0)) or (dy > 0) else -1.0
                        shift = 0.5 * overlap * sign
                        y[j] += shift
                        y[i] -= shift
                        moved = True
            ymin = float(np.nanmin(y_base) - y_pad)
            ymax = float(np.nanmax(y_base) + y_pad)
            y = np.clip(y, ymin, ymax)
            if not moved:
                break
        min_margin = np.inf
        for i in range(n):
            for j in range(i + 1, n):
                dij = float(np.hypot(x[j] - x[i], y[j] - y[i]))
                target = float(r[i] + r[j] + min_gap)
                min_margin = min(min_margin, dij - target)
        df[y_col] = y
        return df, float(min_margin)

def _pair_key(a: str, b: str) -> tuple[str, str]:
    return tuple(sorted((str(a), str(b))))

def _summarize_variables(pair_df: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
    lead_counts = pair_df["leader"].value_counts() if not pair_df.empty else pd.Series(dtype=float)
    follow_counts = pair_df["follower"].value_counts() if not pair_df.empty else pd.Series(dtype=float)
    rows = []
    for v in variables:
        n_lead = int(lead_counts.get(v, 0))
        n_follow = int(follow_counts.get(v, 0))
        n_total = n_lead + n_follow
        lead_tendency = float((n_lead - n_follow) / n_total) if n_total > 0 else np.nan
        involved = pair_df[(pair_df["leader"] == v) | (pair_df["follower"] == v)]
        var_unc = float(involved["uncertainty_norm"].median()) if (not involved.empty and "uncertainty_norm" in involved.columns) else np.nan
        rows.append({
            "variable": v,
            "n_pairs": n_total,
            "lead_tendency": lead_tendency,
            "uncertainty_norm": var_unc,
        })
    return pd.DataFrame(rows)

sat_vars_cmp = sorted(cmp_tbl.loc[cmp_tbl["is_satellite"], "variable"].astype(str).tolist())
all_vars_cmp = sorted(cmp_tbl["variable"].astype(str).tolist())
pair_sat_sat = pair_all_cmp[
    pair_all_cmp["var_x"].astype(str).isin(sat_vars_cmp)
    & pair_all_cmp["var_y"].astype(str).isin(sat_vars_cmp)
]
sat_only_summary = _summarize_variables(pair_sat_sat, sat_vars_cmp).rename(columns={
    "lead_tendency": "lead_tendency_sat_only",
    "uncertainty_norm": "uncertainty_sat_only",
    "n_pairs": "n_pairs_sat_only",
})

anchor_cmp = anchor_base.merge(
    sat_only_summary[["variable", "lead_tendency_sat_only", "uncertainty_sat_only", "n_pairs_sat_only"]],
    on="variable",
    how="left",
)

# If sat-only cannot be estimated for a variable, keep augmented as fallback.
anchor_cmp["lead_tendency_sat_only"] = anchor_cmp["lead_tendency_sat_only"].fillna(anchor_cmp["lead_tendency_aug"])

anchor_cmp["uncertainty_sat_only"] = anchor_cmp["uncertainty_sat_only"].fillna(anchor_cmp["uncertainty_aug"])


def _build_variant(df: pd.DataFrame, sat_mode: str, seed: int) -> tuple[pd.DataFrame, float]:
    out = df.copy()
    if sat_mode == "mixed":
        sat_tendency = out["lead_tendency_aug"]
        sat_unc = out["uncertainty_aug"]
    elif sat_mode == "sat_only":
        sat_tendency = out["lead_tendency_sat_only"]
        sat_unc = out["uncertainty_sat_only"]
    else:
        raise ValueError("sat_mode must be 'mixed' or 'sat_only'")

    out["lead_tendency_anchor_variant"] = np.where(out["is_satellite"], sat_tendency, out["lead_tendency_core"])

    out["uncertainty_anchor_variant"] = np.where(out["is_satellite"], sat_unc, out["uncertainty_core"])

    out["x_anchor"] = -out["lead_tendency_anchor_variant"]
    out["y_anchor"] = out["uncertainty_anchor_variant"]
    out["y_anchor"] = out["y_anchor"].fillna(out["uncertainty_aug"]).fillna(out["y_anchor"].median())

    rng = np.random.default_rng(seed)
    out["y_plot"] = out["y_anchor"] + rng.uniform(-float(STYLE["y_jitter"]), float(STYLE["y_jitter"]), size=len(out))

    out["node_size_plot"] = STYLE["node_size_min"] + STYLE["node_size_span"] * out["n_pairs_aug"] / max(1, out["n_pairs_aug"].max())
    size_norm = out["node_size_plot"] / max(1.0, float(out["node_size_plot"].max()))
    out["radius_hint"] = 0.028 + 0.070 * size_norm

    out, margin = _enforce_min_distance_y(
        out,
        x_col="x_anchor",
        y_col="y_plot",
        radius_col="radius_hint",
        min_gap=float(STYLE["min_node_gap"]),
        iters=int(STYLE["repel_iters"]),
        y_pad=float(STYLE["repel_y_pad"]),
    )
    out["label"] = out["variable"].map(_pretty_name)
    return out, float(margin)

anchor_mixed, margin_mixed = _build_variant(anchor_cmp, sat_mode="mixed", seed=42)
anchor_sat_only, margin_sat = _build_variant(anchor_cmp, sat_mode="sat_only", seed=42)

# Keep same edge set in both panes; this isolates anchor-definition effects.
pair_lookup_cmp = {}
pair_all_ranked = pair_all_cmp.copy()
if "certainty_norm" not in pair_all_ranked.columns:
    pair_all_ranked["certainty_norm"] = 1.0 - pair_all_ranked.get("uncertainty_norm", 0.5)
pair_all_ranked["certainty_norm"] = pair_all_ranked["certainty_norm"].clip(0.0, 1.0)

for _, r in pair_all_ranked.iterrows():
    k = _pair_key(r["var_x"], r["var_y"])
    score = (float(r.get("certainty_norm", 0.0)), float(r.get("median_abs_r", 0.0)))
    if (k not in pair_lookup_cmp) or (score > pair_lookup_cmp[k][0]):
        pair_lookup_cmp[k] = (score, r.copy())

def _select_edges(nodes_plot: pd.DataFrame) -> pd.DataFrame:
    node_names = list(nodes_plot.index)
    selected = set()
    for v in node_names:
        xv = float(nodes_plot.loc[v, "x_anchor"])

        yv = float(nodes_plot.loc[v, "y_plot"])

        drows = []
        for u in node_names:
            if u == v:
                continue
            xu = float(nodes_plot.loc[u, "x_anchor"])

            yu = float(nodes_plot.loc[u, "y_plot"])

            d = np.sqrt((xu - xv) ** 2 + STYLE["distance_y_weight"] * (yu - yv) ** 2)
            drows.append((u, d))
        drows = sorted(drows, key=lambda t: t[1])[: int(STYLE["min_neighbors_per_node"])]
        for u, _ in drows:
            selected.add(_pair_key(v, u))

    extra_added = 0
    ranked = pair_all_ranked.sort_values(["certainty_norm", "median_abs_r"], ascending=[False, False])
    for _, r in ranked.iterrows():
        k = _pair_key(r["var_x"], r["var_y"])

        if k in pair_lookup_cmp and k not in selected:
            selected.add(k)
            extra_added += 1
            if extra_added >= int(STYLE["extra_top_edges"]):
                break

    edges = pd.DataFrame([pair_lookup_cmp[k][1] for k in selected if k in pair_lookup_cmp]).copy()
    edges["source"] = edges["leader"].astype(str)
    edges["target"] = edges["follower"].astype(str)
    edges["edge_width"] = STYLE["edge_width_min"] + (STYLE["edge_width_max"] - STYLE["edge_width_min"]) * edges["certainty_norm"]
    edges["edge_alpha"] = STYLE["edge_alpha_min"] + (STYLE["edge_alpha_max"] - STYLE["edge_alpha_min"]) * edges["certainty_norm"]
    return edges

# Build edge set from mixed layout; reuse in both panes.
edge_cmp = _select_edges(anchor_mixed.set_index("variable"))

def _draw_network(ax, nodes_df: pd.DataFrame, edges_df: pd.DataFrame, title: str):
    nodes_plot = nodes_df.set_index("variable")
    ax.set_facecolor(PALETTE["background"])


    for _, e in edges_df.iterrows():
        s = str(e["source"])

        t = str(e["target"])

        if s not in nodes_plot.index or t not in nodes_plot.index:
            continue
        x1, y1 = float(nodes_plot.loc[s, "x_anchor"]), float(nodes_plot.loc[s, "y_plot"])

        x2, y2 = float(nodes_plot.loc[t, "x_anchor"]), float(nodes_plot.loc[t, "y_plot"])

        moves_to_later = x2 > x1
        color = PALETTE["edge_forward"] if moves_to_later else PALETTE["edge_backward"]
        rad = STYLE["edge_curve_rad"] if (y2 - y1) >= 0 else -STYLE["edge_curve_rad"]
        arrow = FancyArrowPatch(
            (x1, y1), (x2, y2),
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>",
            mutation_scale=8 + 2.2 * float(e["certainty_norm"]),
            linewidth=float(e["edge_width"]),
            color=color,
            alpha=float(e["edge_alpha"]),
            shrinkA=10,
            shrinkB=10,
        )
        ax.add_patch(arrow)

    node_color = np.where(nodes_df["is_satellite"], PALETTE["sat_node"], PALETTE["core_node"])

    ax.scatter(
        nodes_df["x_anchor"], nodes_df["y_plot"],
        s=nodes_df["node_size_plot"],
        c=node_color,
        edgecolors="white",
        linewidths=1.0,
        zorder=3,
    )

    for _, r in nodes_df.iterrows():
        ax.text(
            float(r["x_anchor"]),
            float(r["y_plot"]) + 0.035,
            str(r["label"]),
            ha="center",
            va="bottom",
            fontsize=STYLE["label_fontsize"],
            color=PALETTE["text"],
            bbox={
                "boxstyle": f"round,pad={STYLE['label_box_pad']}",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": STYLE["label_box_alpha"],
            },
        )

    ax.set_xlim(-1.05, 1.05)
    ax.axvline(0, color=PALETTE["reference"], linestyle="--", linewidth=1.0)
    ax.xaxis.grid(True, alpha=0.22)
    ax.yaxis.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(title)
    ax.set_xlabel("Earlier timing tendency   <----->   Later timing tendency")

fig, axes = plt.subplots(1, 2, figsize=(17.2, 8.0), constrained_layout=True)
_draw_network(axes[0], anchor_mixed, edge_cmp, "A) Core-anchored + satellite mixed-system")
_draw_network(axes[1], anchor_sat_only, edge_cmp, "B) Core-anchored + satellite-only")
plt.show()

sat_shift = anchor_mixed.merge(
    anchor_sat_only[["variable", "lead_tendency_anchor_variant"]].rename(columns={"lead_tendency_anchor_variant": "lead_tendency_sat_only_variant"}),
    on="variable",
    how="left",
)
sat_shift = sat_shift[sat_shift["is_satellite"]].copy()
sat_shift["delta_sat_only_minus_mixed"] = sat_shift["lead_tendency_sat_only_variant"] - sat_shift["lead_tendency_anchor_variant"]

print(f"Mixed overlap margin: {margin_mixed:.3f} | Sat-only overlap margin: {margin_sat:.3f}")
print("Satellite tendency shifts (sat-only minus mixed):")
display(
    sat_shift.sort_values("delta_sat_only_minus_mixed", key=lambda s: s.abs(), ascending=False)[
        ["variable", "lead_tendency_anchor_variant", "lead_tendency_sat_only_variant", "delta_sat_only_minus_mixed"]
    ]
)

ANCHOR_VARIABLE_TABLE_MIXED = anchor_mixed.copy()
ANCHOR_VARIABLE_TABLE_SAT_ONLY = anchor_sat_only.copy()
ANCHOR_EDGE_TABLE_COMPARE = edge_cmp.copy()
print("Created comparison tables: ANCHOR_VARIABLE_TABLE_MIXED, ANCHOR_VARIABLE_TABLE_SAT_ONLY, ANCHOR_EDGE_TABLE_COMPARE")

## Change point detection (ruptures)

In [ ]:
raw_pool

In [ ]:
# Single-variable breakpoint test cell (tune parameters here)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ruptures as rpt
except Exception as exc:
    raise ImportError(
        "Package 'ruptures' is required for breakpoint detection. "
        "Install it in this notebook kernel (e.g., pip install ruptures) and rerun. "
        f"Original error: {exc}"
    )

if "ALLPAIR_INPUT_DAILY" not in globals():
    raise ValueError("ALLPAIR_INPUT_DAILY not found. Run the all-pairs preparation cell first.")

# ---------------------------
# User parameters (test mode)
# ---------------------------
CP_METHOD = "pelt"          # "pelt" or "dynp"
CP_MODEL = "l2"            # common: "l2", "rbf", "linear"
CP_PEN = 2.0                # used for PELT
CP_N_BKPS = 4                # used for Dynp
CP_MIN_SIZE = 14             # minimum segment length (days)
CP_JUMP = 1                 # speed/precision tradeoff
CP_NORMALIZE = True          # z-score before detection
CP_STRESS_WINDOW_DAYS = 15   # highlight +- window around stress event

# SG handling: input is already smoothed, so extra smoothing is OFF by default.
CP_USE_SAVGOL_INPUT = True   # map selected variable to *_sg/*_savgol if possible
CP_REQUIRE_SAVGOL = True     # keep only SG columns in options (except default soil column)
CP_APPLY_EXTRA_SMOOTHING = False
CP_SMOOTH_WIN = 9            # only used when CP_APPLY_EXTRA_SMOOTHING=True

# Soil moisture default requested by user (from out_cc).
CP_SOIL_DEFAULT = "sm_pct_mean"

# Variable selection: choose ONE of these (name has priority over index).
CP_VARIABLE = None
CP_VARIABLE_INDEX = 34

daily_cols = [str(c) for c in ALLPAIR_INPUT_DAILY.columns]
out_cc_cols = [str(c) for c in out_cc.columns] if "out_cc" in globals() and isinstance(out_cc, pd.DataFrame) else []
all_cols = sorted(set(daily_cols + out_cc_cols))
sg_suffixes = ("_sg", "_savgol")

def _is_sg_col(v: str) -> bool:
    v = str(v).lower()
    return v.endswith(sg_suffixes)

def _to_sg_col(v: str, columns: list[str]) -> str:
    v = str(v)
    if v in columns and _is_sg_col(v):
        return v

    cands = []
    if v.endswith("_raw"):
        cands.append(v[:-4] + "_sg")
        cands.append(v[:-4] + "_savgol")
    else:
        cands.append(v + "_sg")
        cands.append(v + "_savgol")
        cands.append(v.replace("_raw", "_sg"))
        cands.append(v.replace("_raw", "_savgol"))

    for c in cands:
        if c in columns:
            return c

    return v if v in columns else ""

def _select_series(var_name: str) -> tuple[pd.Series, str]:
    v = str(var_name)
    if v in ALLPAIR_INPUT_DAILY.columns:
        return ALLPAIR_INPUT_DAILY[v], "ALLPAIR_INPUT_DAILY"
    if "out_cc" in globals() and isinstance(out_cc, pd.DataFrame) and v in out_cc.columns:
        return out_cc[v], "out_cc"

    # Try SG mapping fallback
    mapped = _to_sg_col(v, all_cols) if CP_USE_SAVGOL_INPUT else ""
    if mapped in ALLPAIR_INPUT_DAILY.columns:
        return ALLPAIR_INPUT_DAILY[mapped], "ALLPAIR_INPUT_DAILY"
    if mapped and "out_cc" in globals() and isinstance(out_cc, pd.DataFrame) and mapped in out_cc.columns:
        return out_cc[mapped], "out_cc"

    raise ValueError(f"Selected variable '{v}' was not found in ALLPAIR_INPUT_DAILY or out_cc.")

# Build candidate variables from above analysis + explicit soil-moisture columns
base_vars = []
if "ANCHOR_VARIABLE_TABLE" in globals() and not ANCHOR_VARIABLE_TABLE.empty:
    base_vars = ANCHOR_VARIABLE_TABLE["variable"].astype(str).tolist()

soil_re = r"soil|soil_moist|moist|\bsm\b|^sm_|_sm_|swc|vwc|theta|sm_pct_mean"
soil_vars = [
    c for c in all_cols
    if pd.Series([str(c).lower()]).str.contains(soil_re, regex=True).iloc[0]
 ]

raw_pool = sorted(set(base_vars + soil_vars + [c for c in all_cols if _is_sg_col(c)] + [CP_SOIL_DEFAULT]))
if CP_USE_SAVGOL_INPUT:
    mapped = [_to_sg_col(v, all_cols) for v in raw_pool]
    raw_pool = [v for v in mapped if v] + [CP_SOIL_DEFAULT]

if CP_REQUIRE_SAVGOL:
    cp_set = set([v for v in raw_pool if _is_sg_col(v)])
    if CP_SOIL_DEFAULT in all_cols:
        cp_set.add(CP_SOIL_DEFAULT)
    CP_VAR_OPTIONS = sorted(cp_set)
else:
    CP_VAR_OPTIONS = sorted(set(raw_pool))

if not CP_VAR_OPTIONS:
    raise ValueError("No candidate variables found for breakpoint testing.")

print(f"Available variables ({len(CP_VAR_OPTIONS)}):")
for i, v in enumerate(CP_VAR_OPTIONS):
    print(f"[{i:02d}] {v}")

# Resolve selected variable
if CP_VARIABLE is not None:
    CP_VARIABLE = str(CP_VARIABLE)
    if CP_VARIABLE.lower() in {"sm", "soil_moisture", "soil moisture"}:
        if CP_SOIL_DEFAULT in all_cols:
            CP_VARIABLE = CP_SOIL_DEFAULT
        else:
            soil_priority = [
                v for v in CP_VAR_OPTIONS
                if pd.Series([v.lower()]).str.contains(soil_re, regex=True).iloc[0]
            ]
            if not soil_priority:
                raise ValueError("No soil moisture variable found in CP_VAR_OPTIONS.")
            CP_VARIABLE = soil_priority[0]
    elif CP_USE_SAVGOL_INPUT and CP_VARIABLE != CP_SOIL_DEFAULT:
        mapped = _to_sg_col(CP_VARIABLE, all_cols)
        if mapped:
            CP_VARIABLE = mapped

    if CP_VARIABLE not in CP_VAR_OPTIONS:
        near = [v for v in CP_VAR_OPTIONS if CP_VARIABLE.lower() in str(v).lower()]
        msg = f"CP_VARIABLE '{CP_VARIABLE}' is not in CP_VAR_OPTIONS."
        if near:
            msg += f" Close matches: {near[:8]}"
        raise ValueError(msg)
elif CP_VARIABLE_INDEX is not None:
    idx = int(CP_VARIABLE_INDEX)
    if idx < 0 or idx >= len(CP_VAR_OPTIONS):
        raise ValueError(f"CP_VARIABLE_INDEX must be in [0, {len(CP_VAR_OPTIONS)-1}].")
    CP_VARIABLE = CP_VAR_OPTIONS[idx]
else:
    CP_VARIABLE = CP_SOIL_DEFAULT if CP_SOIL_DEFAULT in CP_VAR_OPTIONS else CP_VAR_OPTIONS[0]

selected_series, selected_source = _select_series(CP_VARIABLE)
print(f"Selected variable: {CP_VARIABLE} (source: {selected_source})")
print(
    f"Z-normalization: {'ON' if CP_NORMALIZE else 'OFF'} | "
    f"Extra smoothing: {'ON' if CP_APPLY_EXTRA_SMOOTHING else 'OFF'}"
)

def _resolve_stress_event_date(df_daily: pd.DataFrame) -> pd.Timestamp:
    if "event_date" in globals():
        return pd.Timestamp(event_date).normalize()

    if "SWP_COL" in globals() and SWP_COL in df_daily.columns and "SWP_STRESS_MPA" in globals():
        s = pd.to_numeric(df_daily[SWP_COL], errors="coerce").dropna()
        crossed = s[s <= float(SWP_STRESS_MPA)]
        if not crossed.empty:
            return pd.Timestamp(crossed.index.min()).normalize()

    idx = pd.DatetimeIndex(df_daily.index).sort_values()
    return pd.Timestamp(idx[len(idx) // 2]).normalize()

def detect_breakpoints_series(
    series: pd.Series,
    method: str = "pelt",
    model: str = "rbf",
    pen: float = 6.0,
    n_bkps: int = 4,
    min_size: int = 14,
    jump: int = 1,
    smooth_win: int = 7,
    normalize: bool = True,
    apply_extra_smoothing: bool = False,
 ) -> dict:
    s = pd.to_numeric(series, errors="coerce")
    if not isinstance(s.index, pd.DatetimeIndex):
        try:
            s.index = pd.to_datetime(s.index)
        except Exception:
            pass
    s = s.dropna().sort_index()
    if len(s) < max(30, 2 * min_size + 2):
        return {
            "series_input": s,
            "series_detect": s.copy(),
            "break_dates": [],
            "periods": [],
            "bkps_idx": [],
        }

    if apply_extra_smoothing and int(max(1, smooth_win)) > 1:
        s_detect = s.rolling(int(max(1, smooth_win)), center=True, min_periods=1).median()
    else:
        s_detect = s.copy()

    x = s_detect.to_numpy(dtype=float)
    if normalize and np.nanstd(x) > 0:
        x = (x - np.nanmean(x)) / np.nanstd(x)

    X = x.reshape(-1, 1)
    m = str(method).lower()
    if m == "pelt":
        algo = rpt.Pelt(model=model, min_size=int(min_size), jump=int(jump)).fit(X)
        bkps = algo.predict(pen=float(pen))
    elif m == "dynp":
        algo = rpt.Dynp(model=model, min_size=int(min_size), jump=int(jump)).fit(X)
        bkps = algo.predict(n_bkps=int(n_bkps))
    else:
        raise ValueError("method must be 'pelt' or 'dynp'")

    cp_idx = sorted(set([int(i) - 1 for i in bkps[:-1] if 0 < int(i) < len(s)]))
    break_dates = [pd.Timestamp(s.index[i]).normalize() for i in cp_idx]

    bounds = [0] + [i + 1 for i in cp_idx] + [len(s)]
    periods = []
    for i in range(len(bounds) - 1):
        a, b = bounds[i], bounds[i + 1]
        if b <= a:
            continue
        p0 = pd.Timestamp(s.index[a]).normalize()
        p1 = pd.Timestamp(s.index[b - 1]).normalize()
        periods.append((p0, p1))

    return {
        "series_input": s,
        "series_detect": s_detect,
        "break_dates": break_dates,
        "periods": periods,
        "bkps_idx": cp_idx,
    }

stress_dt = _resolve_stress_event_date(ALLPAIR_INPUT_DAILY)
test_out = detect_breakpoints_series(
    selected_series,
    method=CP_METHOD,
    model=CP_MODEL,
    pen=CP_PEN,
    n_bkps=CP_N_BKPS,
    min_size=CP_MIN_SIZE,
    jump=CP_JUMP,
    smooth_win=CP_SMOOTH_WIN,
    normalize=CP_NORMALIZE,
    apply_extra_smoothing=CP_APPLY_EXTRA_SMOOTHING,
)

s_input = test_out["series_input"]
s_detect = test_out["series_detect"]
break_dates = test_out["break_dates"]
periods = test_out["periods"]

if s_input.empty:
    raise ValueError(f"Variable {CP_VARIABLE} has no valid observations after numeric conversion.")

input_label = "Input series (SG column)" if _is_sg_col(CP_VARIABLE) else "Input series (native column)"
fig, ax = plt.subplots(figsize=(13.2, 6.6))
ax.plot(s_input.index, s_input.values, color="#9aa0a6", linewidth=1.2, alpha=0.75, label=input_label)
if CP_APPLY_EXTRA_SMOOTHING:
    ax.plot(s_detect.index, s_detect.values, color="#1f77b4", linewidth=2.0, alpha=0.95, label="Extra smoothed (rolling median)")

for i, (p0, p1) in enumerate(periods):
    if i % 2 == 0:
        ax.axvspan(p0, p1, color="#eef3f8", alpha=0.9, zorder=0)

for dt in break_dates:
    ax.axvline(dt, color="#d55e00", linestyle="--", linewidth=1.5, alpha=0.9)

ax.axvline(stress_dt, color="#111827", linestyle="-", linewidth=2.2, alpha=0.85, label="Stress event")
ax.axvspan(
    stress_dt - pd.Timedelta(days=int(CP_STRESS_WINDOW_DAYS)),
    stress_dt + pd.Timedelta(days=int(CP_STRESS_WINDOW_DAYS)),
    color="#fbe7e7",
    alpha=0.35,
    zorder=0,
    label=f"Stress window ±{int(CP_STRESS_WINDOW_DAYS)} d",
)

ax.set_title(f"Breakpoint test: {CP_VARIABLE} | method={CP_METHOD}, model={CP_MODEL}")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.grid(alpha=0.22)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

period_table = pd.DataFrame(
    [
        {"period_id": i + 1, "start": p0, "end": p1, "days": int((p1 - p0).days + 1)}
        for i, (p0, p1) in enumerate(periods)
    ]
)
break_table = pd.DataFrame({"breakpoint": break_dates})
if not break_table.empty:
    break_table["days_from_stress"] = (break_table["breakpoint"] - stress_dt).dt.days
    break_table["abs_days_from_stress"] = break_table["days_from_stress"].abs()
    break_table = break_table.sort_values("abs_days_from_stress")

print(f"Stress event date: {stress_dt.date()} | Variable tested: {CP_VARIABLE}")
print(f"Detected breakpoints: {len(break_dates)}")
display(break_table)
display(period_table)

# Export helpers for the automation cell
CP_HELPER_DETECT_BREAKPOINTS = detect_breakpoints_series
CP_HELPER_STRESS_DATE = stress_dt
CP_HELPER_VAR_OPTIONS = CP_VAR_OPTIONS

In [ ]:
# Automatic breakpoint scan for selected variables + stress-nearest onset-window Gantt plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D

try:
    import ruptures as rpt  # noqa: F401
except Exception as exc:
    raise ImportError(
        "Package 'ruptures' is required for breakpoint detection. "
        "Install it in this notebook kernel (e.g., pip install ruptures) and rerun. "
        f"Original error: {exc}"
    )

if "ALLPAIR_INPUT_DAILY" not in globals():
    raise ValueError("ALLPAIR_INPUT_DAILY not found. Run the all-pairs preparation cell first.")
if "CP_HELPER_DETECT_BREAKPOINTS" not in globals():
    raise ValueError("Run Cell 36 first to define breakpoint helper and stress date.")

# ---------------------------
# User parameters (batch mode)
# ---------------------------
AUTO_METHOD = "pelt"
AUTO_MODEL = "l2"
AUTO_PEN = 2.0          # default penalty for all variables
AUTO_N_BKPS = 4
AUTO_MIN_SIZE = 14
AUTO_JUMP = 1
AUTO_SMOOTH_WIN = 1
AUTO_NORMALIZE = True

# Per-variable penalty overrides (takes precedence over AUTO_PEN)
AUTO_PEN_OVERRIDE = {
    "et_mmd_mean_sg": 1.0,
    "sapflow_jscm3cm2d_mean_sg": 1.0,
}

AUTO_INCLUDE_ONLY_CP_OPTIONS = True
AUTO_MIN_VALID_POINTS = 80
AUTO_REQUIRE_SOIL_MOISTURE = False
AUTO_REQUIRE_SG = True
AUTO_DROP_STD = True
AUTO_SOIL_START_BEFORE_MONTH = 8
AUTO_SOIL_DEFAULT = "sm_pct_mean"

AUTO_SELECTED_VARIABLES = [
    "et_mmd_mean_sg", "gcc_p90_mean_sg", "gpp_umolm2s_mean_sg",
    "leaf_angle_cam65_mean_sg", "leaf_angle_cam67_mean_sg", "pai_total_sg",
    "s1_cr_savgol", "s1_span_savgol",
    "s2_cire_savgol", "s2_evi_savgol", "s2_kndvi_savgol", "s2_mtci_savgol",
    "s2_ndii_savgol", "s2_ndvi_savgol", "s2_nirv_savgol", "s2_savi_savgol",
    "sapflow_jscm3cm2d_mean_sg", "sm_pct_mean_sg", "swp_mpa_predawn_mean_sg",
    "tair_c_mean_sg", "twd_um_mean_sg", "vod_mean_sg", "vpd_hpa_max_sg",
]

# ──────────────────────────────────────────────────────
# Display names and SPAC compartment mapping
# ──────────────────────────────────────────────────────
_DISP_NAME = {
    "tair_c_mean_sg":              r"$T_\mathrm{air}$",
    "vpd_hpa_max_sg":              r"$\mathrm{VPD}_\mathrm{max}$",
    "sm_pct_mean_sg":              r"$\theta_\mathrm{soil}$",
    "swp_mpa_predawn_mean_sg":     r"$\Psi_\mathrm{stem}$ (pre-dawn)",
    "sapflow_jscm3cm2d_mean_sg":   r"$J_s$",
    "twd_um_mean_sg":              r"TWD",
    "vod_mean_sg":                 r"VOD",
    "leaf_angle_cam65_mean_sg":    r"Leaf angle (3.5 m)",
    "leaf_angle_cam67_mean_sg":    r"Leaf angle (8.5 m)",
    "pai_total_sg":                r"PAI",
    "et_mmd_mean_sg":              r"ET",
    "gcc_p90_mean_sg":             r"GCC",
    "gpp_umolm2s_mean_sg":         r"GPP",
    "s1_cr_savgol":                r"S1  $C_R$",
    "s1_span_savgol":              r"S1  SPAN",
    "s2_cire_savgol":              r"S2  CI$_\mathrm{re}$",
    "s2_evi_savgol":               r"S2  EVI",
    "s2_kndvi_savgol":             r"S2  kNDVI",
    "s2_mtci_savgol":              r"S2  MTCI",
    "s2_ndii_savgol":              r"S2  NDII",
    "s2_ndvi_savgol":              r"S2  NDVI",
    "s2_nirv_savgol":              r"S2  NIR$_v$",
    "s2_savi_savgol":              r"S2  SAVI",
}

_COMPARTMENT_MAP = {
    "tair_c_mean_sg":              "Atmosphere",
    "vpd_hpa_max_sg":              "Atmosphere",
    "sm_pct_mean_sg":              "Soil water",
    "swp_mpa_predawn_mean_sg":     "Soil water",
    "sapflow_jscm3cm2d_mean_sg":   "Plant hydraulics",
    "twd_um_mean_sg":              "Plant hydraulics",
    "vod_mean_sg":                 "Canopy & water content",
    "leaf_angle_cam65_mean_sg":    "Canopy & water content",
    "leaf_angle_cam67_mean_sg":    "Canopy & water content",
    "pai_total_sg":                "Canopy & water content",
    "et_mmd_mean_sg":              "Ecosystem fluxes",
    "gcc_p90_mean_sg":             "Ecosystem fluxes",
    "gpp_umolm2s_mean_sg":         "Ecosystem fluxes",
    "s1_cr_savgol":                "Satellite",
    "s1_span_savgol":              "Satellite",
    "s2_cire_savgol":              "Satellite",
    "s2_evi_savgol":               "Satellite",
    "s2_kndvi_savgol":             "Satellite",
    "s2_mtci_savgol":              "Satellite",
    "s2_ndii_savgol":              "Satellite",
    "s2_ndvi_savgol":              "Satellite",
    "s2_nirv_savgol":              "Satellite",
    "s2_savi_savgol":              "Satellite",
}

_COMP_ORDER = [
    "Atmosphere",
    "Soil water",
    "Plant hydraulics",
    "Canopy & water content",
    "Ecosystem fluxes",
    "Satellite",
]

# Colours from user-provided palette
# ["#a68a59","#94a659","#66a659","#59a67b","#59a3a6","#5975a6","#6b59a6","#9959a6","#a65984","#a65c59"]
_COMP_COLOR = {
    "Atmosphere":            "#a68a59",   # warm gold – sun/air
    "Soil water":            "#59a67b",   # teal-green – moist earth
    "Plant hydraulics":      "#5975a6",   # blue – water transport
    "Canopy & water content":"#66a659",   # green – leaves
    "Ecosystem fluxes":      "#94a659",   # olive-yellow – photosynthesis
    "Satellite":             "#6b59a6",   # purple – remote sensing
}

# ──────────────────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────────────────
soil_re = r"soil|soil_moist|moist|\bsm\b|^sm_|_sm_|swc|vwc|theta|sm_pct_mean"
sg_suffixes = ("_sg", "_savgol")

def _to_naive_ts(x) -> pd.Timestamp:
    ts = pd.Timestamp(x)
    if ts.tzinfo is not None:
        ts = ts.tz_localize(None)
    return ts.normalize()

def _is_sg(v: str) -> bool:
    return str(v).lower().endswith(sg_suffixes)

def _is_std(v: str) -> bool:
    vv = str(v).lower()
    return ("_std" in vv) or vv.endswith("std")

def _is_soil(v: str) -> bool:
    return bool(pd.Series([str(v).lower()]).str.contains(soil_re, regex=True).iloc[0])

def _resolve_series(v: str) -> tuple[pd.Series, str]:
    v = str(v)
    if v in ALLPAIR_INPUT_DAILY.columns:
        s = pd.to_numeric(ALLPAIR_INPUT_DAILY[v], errors="coerce")
        s.index = pd.to_datetime(s.index)
        if getattr(s.index, "tz", None) is not None:
            s.index = s.index.tz_localize(None)
        return s, "ALLPAIR_INPUT_DAILY"
    if "out_cc" in globals() and isinstance(out_cc, pd.DataFrame) and v in out_cc.columns:
        s = pd.to_numeric(out_cc[v], errors="coerce")
        s.index = pd.to_datetime(s.index)
        if getattr(s.index, "tz", None) is not None:
            s.index = s.index.tz_localize(None)
        return s, "out_cc"
    raise KeyError(v)

# ──────────────────────────────────────────────────────
# Build variable pool
# ──────────────────────────────────────────────────────
if AUTO_INCLUDE_ONLY_CP_OPTIONS and "CP_HELPER_VAR_OPTIONS" in globals():
    var_pool = list(pd.unique(pd.Series(list(CP_HELPER_VAR_OPTIONS) + [AUTO_SOIL_DEFAULT])))
else:
    var_pool = list(pd.unique(pd.Series(list(ALLPAIR_INPUT_DAILY.columns) + [AUTO_SOIL_DEFAULT])))

resolved = []
for v in var_pool:
    try:
        _, src = _resolve_series(v)
        resolved.append((str(v), src))
    except Exception:
        pass

if AUTO_REQUIRE_SG:
    resolved = [(v, src) for v, src in resolved if _is_sg(v) or v == AUTO_SOIL_DEFAULT]
if AUTO_DROP_STD:
    resolved = [(v, src) for v, src in resolved if not _is_std(v)]
if AUTO_REQUIRE_SOIL_MOISTURE:
    resolved = [(v, src) for v, src in resolved if _is_soil(v)]

seen = set()
resolved_unique = []
for v, src in resolved:
    if v not in seen:
        resolved_unique.append((v, src))
        seen.add(v)
resolved = resolved_unique

if not resolved:
    raise ValueError("No variables available after SG/std/soil filters.")

print(f"Available variables after filters ({len(resolved)}):")
for i, (v, src) in enumerate(resolved):
    print(f"[{i:02d}] {v}  (source={src})")

if AUTO_SELECTED_VARIABLES:
    wanted = [str(v) for v in AUTO_SELECTED_VARIABLES]
    missing = [v for v in wanted if v not in [x for x, _ in resolved]]
    if missing:
        raise ValueError(f"AUTO_SELECTED_VARIABLES contains unknown names: {missing}")
    keep = set(wanted)
    resolved = [(v, src) for v, src in resolved if v in keep]

if not resolved:
    raise ValueError("No variables left after AUTO_SELECTED_VARIABLES filtering.")

# ──────────────────────────────────────────────────────
# Breakpoint detection
# ──────────────────────────────────────────────────────
stress_dt = _to_naive_ts(CP_HELPER_STRESS_DATE)
soil_cutoff = pd.Timestamp(year=stress_dt.year, month=int(AUTO_SOIL_START_BEFORE_MONTH), day=1).normalize()

records = []
fallback_nonsoil_count = 0
for v, src in resolved:
    s, _src = _resolve_series(v)
    s = s.dropna().sort_index()
    if len(s) < int(AUTO_MIN_VALID_POINTS):
        continue

    pen = AUTO_PEN_OVERRIDE.get(v, AUTO_PEN)

    out = CP_HELPER_DETECT_BREAKPOINTS(
        s,
        method=AUTO_METHOD,
        model=AUTO_MODEL,
        pen=pen,
        n_bkps=AUTO_N_BKPS,
        min_size=AUTO_MIN_SIZE,
        jump=AUTO_JUMP,
        smooth_win=AUTO_SMOOTH_WIN,
        normalize=AUTO_NORMALIZE,
        apply_extra_smoothing=False,
    )

    bdates = sorted([_to_naive_ts(d) for d in out["break_dates"]])
    if len(bdates) < 2:
        continue

    valid_starts = [d for d in bdates if any(x > d for x in bdates)]
    if not valid_starts:
        continue

    soil_flag = _is_soil(v)
    if soil_flag:
        starts = [d for d in valid_starts if d < soil_cutoff]
        if starts:
            start = starts[-1]
        else:
            before_stress = [d for d in valid_starts if d <= stress_dt]
            start = before_stress[-1] if before_stress else min(valid_starts, key=lambda d: abs((d - stress_dt).days))
    else:
        starts_after_cutoff = [d for d in valid_starts if d >= soil_cutoff]
        if starts_after_cutoff:
            start = min(starts_after_cutoff, key=lambda d: abs((d - stress_dt).days))
        else:
            start = min(valid_starts, key=lambda d: abs((d - stress_dt).days))
            fallback_nonsoil_count += 1

    next_after = [d for d in bdates if d > start]
    if not next_after:
        continue
    end = next_after[0]

    before = [d for d in bdates if d <= stress_dt]
    after  = [d for d in bdates if d > stress_dt]
    nearest_before = max(before) if before else pd.NaT
    nearest_after  = min(after)  if after  else pd.NaT

    records.append({
        "variable":              str(v),
        "source":                _src,
        "n_points":              int(len(s)),
        "n_breakpoints":         int(len(bdates)),
        "bp_start":              start,
        "bp_end":                end,
        "onset":                 start,
        "window_start":          start,
        "window_end":            end,
        "window_days":           int((end - start).days),
        "nearest_before":        nearest_before,
        "nearest_after":         nearest_after,
        "d_start_to_stress_days":int((start - stress_dt).days),
        "d_end_to_stress_days":  int((end - stress_dt).days),
        "soil_moisture_like":    soil_flag,
        "is_sg_column":          _is_sg(v),
        "pen_used":              pen,
    })

if not records:
    raise ValueError(
        "No variables produced valid start/end windows under current settings. "
        "Try reducing min_size or penalty."
    )

bp_auto = pd.DataFrame(records).sort_values(["onset", "variable"]).reset_index(drop=True)

# ──────────────────────────────────────────────────────
# Sort by SPAC compartment, then by onset within compartment
# ──────────────────────────────────────────────────────
_COMP_ORDER_NUM = {c: k for k, c in enumerate(_COMP_ORDER)}
bp_auto["compartment"] = bp_auto["variable"].map(_COMPARTMENT_MAP).fillna("Other")
bp_auto["comp_order"]  = bp_auto["compartment"].map(_COMP_ORDER_NUM).fillna(99).astype(int)
bp_auto = bp_auto.sort_values(["comp_order", "onset"]).reset_index(drop=True)
bp_auto["label"] = bp_auto["variable"].map(_DISP_NAME).fillna(bp_auto["variable"])

n_vars = len(bp_auto)

# ══════════════════════════════════════════════════════
# Poster-ready Gantt chart
# ══════════════════════════════════════════════════════
plt.rcParams.update({
    "font.family":        "sans-serif",
    "mathtext.fontset":   "dejavusans",
    "axes.linewidth":     0.8,
    "xtick.major.width":  0.8,
})

fig_h = max(11, 0.60 * n_vars + 3.2)
fig, ax = plt.subplots(figsize=(18, fig_h), facecolor="white")
ax.set_facecolor("white")

# y limits (inverted: row 0 at top)
ax.set_ylim(n_vars - 0.5, -0.5)

# x limits
_all_dt  = list(bp_auto["window_start"]) + list(bp_auto["window_end"]) + [stress_dt]
x_min_dt = min(_all_dt) - pd.Timedelta(days=14)
x_max_dt = max(_all_dt) + pd.Timedelta(days=14)
x_min_num = mdates.date2num(x_min_dt.to_pydatetime())
x_max_num = mdates.date2num(x_max_dt.to_pydatetime())
ax.set_xlim(x_min_num, x_max_num)

# ── Compartment background bands (very subtle tint) ──
for comp in _COMP_ORDER:
    _rows = bp_auto[bp_auto["compartment"] == comp]
    if _rows.empty:
        continue
    y0 = _rows.index.min() - 0.49
    y1 = _rows.index.max() + 0.49
    ax.axhspan(y0, y1, color=_COMP_COLOR[comp], alpha=0.07, zorder=0)

# Alternating row highlight
for i in range(n_vars):
    if i % 2 == 0:
        ax.axhspan(i - 0.49, i + 0.49, color="#f4f4f4", alpha=0.50, zorder=0)

# ── Stress-event shading ──────────────────────────────
_x_lo = mdates.date2num((stress_dt - pd.Timedelta(days=14)).to_pydatetime())
_x_hi = mdates.date2num((stress_dt + pd.Timedelta(days=14)).to_pydatetime())
ax.axvspan(_x_lo, _x_hi, color="#fde8e8", alpha=0.65, zorder=1)

# ── Horizontal bars ───────────────────────────────────
_bar_h = 0.54
for i, row in bp_auto.iterrows():
    c_bar = _COMP_COLOR.get(row["compartment"], "#888888")
    x_s = mdates.date2num(pd.Timestamp(row["window_start"]).to_pydatetime())
    x_e = mdates.date2num(pd.Timestamp(row["window_end"]).to_pydatetime())

    # Main bar
    ax.add_patch(Rectangle(
        (x_s, i - _bar_h / 2), x_e - x_s, _bar_h,
        facecolor=c_bar, edgecolor="white", linewidth=1.0,
        alpha=0.80, zorder=3,
    ))

    # Breakpoint cap lines (slightly darker, extend just beyond bar height)
    c_dark = mcolors.to_hex(tuple(max(0.0, v - 0.22) for v in mcolors.to_rgb(c_bar)))
    for x_bp in (x_s, x_e):
        ax.vlines(x_bp, i - _bar_h / 2 - 0.10, i + _bar_h / 2 + 0.10,
                  colors=c_dark, linewidth=2.4, zorder=4)

# ── Compartment separator lines ───────────────────────
_prev_comp = None
for i, row in bp_auto.iterrows():
    if row["compartment"] != _prev_comp and i > 0:
        ax.axhline(i - 0.5, color="#cccccc", linewidth=1.0, zorder=2)
    _prev_comp = row["compartment"]

# ── Stress-event vertical line + annotation ──────────
_x_stress = mdates.date2num(stress_dt.to_pydatetime())
ax.axvline(_x_stress, color="#c0392b", linewidth=2.0,
           linestyle="--", alpha=0.88, zorder=5)
ax.text(_x_stress, 1.013,
        f"Stress event  ({stress_dt.strftime('%d %b %Y')})",
        transform=ax.get_xaxis_transform(),
        ha="center", va="bottom", fontsize=12,
        color="#c0392b", style="italic", clip_on=False)

# ── X axis ────────────────────────────────────────────
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
ax.tick_params(axis="x", which="major", labelsize=13, length=6, pad=7)
ax.tick_params(axis="x", which="minor", length=3, color="#bbbbbb")

# ── Y axis ────────────────────────────────────────────
ax.set_yticks(range(n_vars))
ax.set_yticklabels(bp_auto["label"].tolist(), fontsize=14)
ax.tick_params(axis="y", length=0, pad=8)

# ── Grid ──────────────────────────────────────────────
ax.grid(axis="x", which="major", alpha=0.18, color="#7788aa", linewidth=0.7, zorder=0)
ax.grid(axis="x", which="minor", alpha=0.07, color="#7788aa", linewidth=0.5, zorder=0)

# ── Labels & title ────────────────────────────────────
ax.set_xlabel("Date", fontsize=15, labelpad=10)
ax.set_title(
    "Breakpoint onset windows relative to stress event",
    fontsize=17, fontweight="bold", pad=20,
)

# ── Spines ────────────────────────────────────────────
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_alpha(0.35)

# ── Compartment labels (right margin) ─────────────────
# axes-y fraction with inverted y: data y=0 → axes y≈1, data y=n-1 → axes y≈0
for comp in _COMP_ORDER:
    _rows = bp_auto[bp_auto["compartment"] == comp]
    if _rows.empty:
        continue
    _y_mid  = (_rows.index.min() + _rows.index.max()) / 2.0
    _y_frac = (n_vars - 0.5 - _y_mid) / n_vars
    ax.text(
        1.007, _y_frac, comp,
        transform=ax.transAxes,
        va="center", ha="left",
        fontsize=12, color=_COMP_COLOR[comp],
        fontweight="bold", clip_on=False,
    )

# ── Legend ────────────────────────────────────────────
_lh = [
    Patch(facecolor=_COMP_COLOR[c], edgecolor="none", alpha=0.80, label=c)
    for c in _COMP_ORDER if c in bp_auto["compartment"].values
]
_lh.append(Line2D([0], [0], color="#c0392b", lw=2.0, linestyle="--", label="Stress event"))
ax.legend(
    handles=_lh, loc="lower left",
    frameon=True, framealpha=0.93, edgecolor="#d0d0d0",
    fontsize=12, title="SPAC compartment", title_fontsize=12,
    handlelength=1.6, handleheight=1.2,
)

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.show()

# ──────────────────────────────────────────────────────
print(f"Stress event date: {stress_dt.date()} | Variables plotted: {len(bp_auto)}")
print(f"Method={AUTO_METHOD}, model={AUTO_MODEL}, default pen={AUTO_PEN}, n_bkps={AUTO_N_BKPS}, min_size={AUTO_MIN_SIZE}")
if AUTO_PEN_OVERRIDE:
    print(f"Per-variable penalty overrides: {AUTO_PEN_OVERRIDE}")
print("SG inputs, no *_std, non-soil starts on/after 01.08, sorted by SPAC compartment then onset.")
if fallback_nonsoil_count > 0:
    print(f"Note: {fallback_nonsoil_count} non-soil variable(s) had no valid post-01.08 start; fallback closest start was used.")
if AUTO_SELECTED_VARIABLES:
    print(f"Manual selection used: {AUTO_SELECTED_VARIABLES}")

display(
    bp_auto[[
        "variable", "label", "compartment", "source", "n_points", "n_breakpoints",
        "bp_start", "bp_end", "window_days",
        "d_start_to_stress_days", "d_end_to_stress_days",
        "nearest_before", "nearest_after",
        "soil_moisture_like", "is_sg_column", "pen_used",
    ]]
)

BREAKPOINT_AUTO_TABLE = bp_auto.copy()
print("Created table: BREAKPOINT_AUTO_TABLE")

# Save the plot
fig.savefig("breakpoint_onset_gantt.pdf", dpi=300, bbox_inches="tight")

In [ ]:
records